# Historische Daten: Kriegsstatus, Kontinente und Extremfälle

**Start:** Dieses Notebook neben deinen vorhandenen Ordner `Data/` legen. Die zehn Originaldateien behalten ihre bisherigen Namen ohne `(1)`. In VS Code den Python-Kernel deiner `.venv` auswählen und **Run All** ausführen. Kein anderes Notebook muss vorher laufen.

**Pakete:** Bei Bedarf in einer eigenen Zelle `%pip install pandas numpy scipy matplotlib openpyxl xlrd pillow` ausführen.

**Ein Ergebnisdokument:** `Ergebnisse/Analysebericht_Krieg_Kontinente_<Zeitstempel>.xlsx`. Alle bisherigen und neuen Grafiken sind eingebettet und nach Hypothese und Gruppierung geordnet. Direkt unter jeder Grafik stehen Kennzahlen. Gruppen, obere/untere Extremfälle, kurze Insights, Modelle und Quellen stehen zusätzlich in filterbaren Tabellen. Diese **eine Excel-Datei** kannst du anschließend zur Auswertung hochladen. Pro Lauf entsteht ein neuer Zeitstempel; die Originaldaten werden nicht verändert.

**Umfang:** H1, H2, H3, H4, H8 sowie die fünf explorativen Fragen E1–E5; sechs Kontinente, exklusive Kontinentkombinationen für Paare, Sensitivität gegenüber Kriegshäufigkeit und Extremfällen. Für die militärbezogenen Fragen H1/H2/H3/E2 zusätzlich Kriegsgruppen, Kriegsstatus innerhalb jedes Kontinents und Regressionsmodelle mit Interaktionen.

**Kriegsbegriff:** Die vorhandene COW-Datei deckt zwischenstaatliche Kriege bis **2007** ab. Ab 2008 ist der Status unbekannt. Keine erfasste Beteiligung bedeutet nicht allgemein Frieden; insbesondere fehlen Bürgerkriege. Zwei beteiligte Staaten müssen nicht gegeneinander kämpfen.

**Eigene Filter:** Im Abschnitt „Frei wählbare Stichproben“ kannst du Kontinent, Kriegsstatus und auszuschließende Länder auswählen. `WAR_CONFIG` steuert unter anderem die Mindestzahl beobachteter Jahre und eine optionale ISO-Länderausschlussliste. Häufigkeitsfilter beziehen sich auf den gemeinsamen bekannten Zeitraum und sind rückblickende Vergleiche. Extremwerte werden in der Hauptanalyse beibehalten.

**Laufzeit:** Die vollständige Aufbereitung, viele Gruppen und rund 150 Grafiken können mehrere Minuten benötigen. Standard-p-Werte ignorieren die Abhängigkeit wiederholter Länder-/Paarjahre; Zusatzmodelle und Gruppenvergleiche bleiben explorativ. H8 unterschreitet bereits insgesamt die bisherige 5.000-Beobachtungen-Vorgabe und bleibt ergänzend.

## Teil A: Bestehende Datenaufbereitung


In [ ]:
# Bei fehlenden Paketen diese Zeile in Jupyter entkommentieren:
# %pip install pandas numpy openpyxl xlrd

from pathlib import Path
import hashlib
import re
import unicodedata
import importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

required_packages = ['openpyxl', 'xlrd']
missing_packages = [p for p in required_packages if importlib.util.find_spec(p) is None]
if missing_packages:
    raise ImportError('Bitte zuerst installieren: ' + ', '.join(missing_packages))

PROJECT_DIR = Path.cwd()  # Notebook liegt auf derselben Ebene wie Data/
DATA_DIR = PROJECT_DIR / 'Data'
START_YEAR, END_YEAR = 1950, 2014
DISTANCE_COLUMN = 'distw'  # bevölkerungsgewichtete geografische Distanz, km

# Qualitätsschwellen sind Annahmen, keine Garantie vollständiger Handelsdaten.
# Kein Ergebnis wird anhand seines Korrelationsvorzeichens ausgewählt.
COVERAGE_MIN, COVERAGE_MAX = 0.90, 1.10
DISTANCE_COVERAGE_MIN = 0.95
MIN_PAIRS = 5000
EXPORT_PREPARED = False  # True schreibt CSV-Dateien nach Prepared/; Originale bleiben erhalten.

FILES = {
    'nmc': 'NMC-70-abridged.csv',
    'national_trade': 'National_COW_4.0.csv',
    'dyadic_trade': 'Dyadic_COW_4.0.csv',
    'state_list': 'statelist2024.csv',
    'geography': 'geo_cepii.xls',
    'distance': 'dist_cepii.xls',
    'maddison': 'mpd2023_web.xlsx',
    'sipri': 'SIPRI-Milex-data-1949-2025_v1.2.xlsx',
    'pwt': 'pwt110.xlsx',
    'wars': 'Inter-StateWarData_v4.0.csv',
}
missing_files = [name for name in FILES.values() if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Gesucht wird in {DATA_DIR.resolve()}. Fehlende Dateien: {missing_files}')

pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 20)
print('Datenordner:', DATA_DIR.resolve())


## 2. Alle Originaldaten importieren

`raw` enthält die unveränderten eingelesenen Tabellen. SIPRI wird zunächst mit allen Textmarkierungen gelesen. PWT verwendet das Blatt `Data`, Maddison `Full data`; regionale Summen werden nicht mit Ländern vermischt. `metadata` bewahrt Legenden und Quellenhinweise auf. Hashwerte helfen, genau die verwendeten Dateien zu dokumentieren.


In [ ]:
raw = {}
for key in ['nmc', 'national_trade', 'dyadic_trade', 'state_list', 'wars']:
    raw[key] = pd.read_csv(DATA_DIR / FILES[key], low_memory=False)
raw['geography'] = pd.read_excel(DATA_DIR / FILES['geography'], engine='xlrd')
raw['distance'] = pd.read_excel(DATA_DIR / FILES['distance'], engine='xlrd')
raw['pwt'] = pd.read_excel(DATA_DIR / FILES['pwt'], sheet_name='Data', engine='openpyxl')
raw['maddison'] = pd.read_excel(DATA_DIR / FILES['maddison'], sheet_name='Full data', engine='openpyxl')
raw['sipri'] = pd.read_excel(DATA_DIR / FILES['sipri'], sheet_name='Constant (2024) US$', header=5, engine='openpyxl')
metadata = {
    'pwt_legend': pd.read_excel(DATA_DIR / FILES['pwt'], sheet_name='Legend'),
    'pwt_info': pd.read_excel(DATA_DIR / FILES['pwt'], sheet_name='Info', header=None),
    'maddison_notes': pd.read_excel(DATA_DIR / FILES['maddison'], sheet_name='Notes', header=None),
    'maddison_sources': pd.read_excel(DATA_DIR / FILES['maddison'], sheet_name='Sources', header=None),
    'sipri_footnotes': pd.read_excel(DATA_DIR / FILES['sipri'], sheet_name='Footnotes', header=None),
}

def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

source_catalog = pd.DataFrame([
    {'dataset': key, 'filename': filename, 'rows': len(raw[key]),
     'columns': len(raw[key].columns), 'sha256': sha256_file(DATA_DIR / filename)}
    for key, filename in FILES.items()
])
display(source_catalog[['dataset', 'filename', 'rows', 'columns']])


## 3. Hilfsfunktionen und Bereinigung

Negative fehlende-Werte-Codes werden **nur in dokumentierten Messwertfeldern** entfernt. Echte Nullwerte bleiben erhalten. Kennungen, Quellencodes und Kategorien werden nicht pauschal umcodiert. Negative BIP-/Bevölkerungswerte sind ungültig; jährliche Wachstumsraten dürfen später negativ sein.

Primärschlüssel werden geprüft. Das Notebook stoppt bei widersprüchlichen Duplikaten, statt sie stillschweigend zu löschen.


In [ ]:
def require_unique(df, keys, label):
    duplicates = df.duplicated(keys, keep=False)
    if duplicates.any():
        display(df.loc[duplicates, keys].head(10))
        raise ValueError(f'{label}: {int(duplicates.sum())} doppelte Schlüssel für {keys}.')

def numeric_measure(df, columns, nonnegative=True):
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if nonnegative:
            result.loc[result[col] < 0, col] = np.nan
    return result

def normalize_name(name):
    if pd.isna(name):
        return ''
    name = unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '', name.lower().replace('&', 'and'))

nmc = numeric_measure(raw['nmc'], ['milex', 'milper', 'irst', 'pec', 'tpop', 'upop', 'cinc'])
national_trade = numeric_measure(raw['national_trade'], ['imports', 'exports', 'alt_imports', 'alt_exports'])
dyadic_trade = numeric_measure(raw['dyadic_trade'], ['flow1', 'flow2', 'smoothflow1', 'smoothflow2', 'smoothtotrade',
    'bel_lux_alt_flow1','bel_lux_alt_flow2','china_alt_flow1','china_alt_flow2'])
state_list = raw['state_list'].copy()
pwt = numeric_measure(raw['pwt'], ['rgdpo', 'rgdpe', 'cgdpe', 'pop', 'emp'])
pwt = pwt.rename(columns={'countrycode': 'iso3', 'country': 'pwt_country'})
maddison = numeric_measure(raw['maddison'], ['gdppc', 'pop'])
maddison = maddison.rename(columns={'countrycode': 'iso3', 'country': 'maddison_country',
                                    'pop': 'maddison_population_thousands'})
geo_locations = raw['geography'].copy()  # Mehrere Städte je Land sind absichtlich vorhanden.
distances = numeric_measure(raw['distance'], ['dist', 'distcap', 'distw', 'distwces'])
distances = distances.rename(columns={'iso_o': 'iso3_origin', 'iso_d': 'iso3_destination'})
for df in [pwt, maddison, geo_locations]:
    df['iso3'] = df['iso3'].astype('string').str.strip().str.upper()
for col in ['iso3_origin', 'iso3_destination']:
    distances[col] = distances[col].astype('string').str.strip().str.upper()

# Echte Umbenennungen desselben Landes, keine Zuordnung von Nachfolgestaaten.
# YUG wird ausdrücklich NICHT zu SRB, SUN NICHT zu RUS.
CEPII_CODE_RENAMES = {'ROM':'ROU','TMP':'TLS','ZAR':'COD'}
geo_locations['cepii_iso3_original'] = geo_locations['iso3']
geo_locations['iso3'] = geo_locations['iso3'].replace(CEPII_CODE_RENAMES)
for col in ['iso3_origin','iso3_destination']:
    distances['cepii_'+col+'_original'] = distances[col]
    distances[col] = distances[col].replace(CEPII_CODE_RENAMES)

for df, keys, label in [
    (nmc, ['ccode', 'year'], 'NMC'),
    (national_trade, ['ccode', 'year'], 'National trade'),
    (dyadic_trade, ['ccode1', 'ccode2', 'year'], 'Dyadic trade'),
    (pwt, ['iso3', 'year'], 'PWT'),
    (maddison, ['iso3', 'year'], 'Maddison'),
    (distances, ['iso3_origin', 'iso3_destination'], 'CEPII distances')]:
    require_unique(df, keys, label)

# Eine Länderübersicht ohne willkürliche Wahl einer Hauptstadtkoordinate.
# Stadtkoordinaten bleiben ausschließlich in geo_locations.
geo_country_fields = ['iso3', 'country', 'area', 'landlocked', 'continent']
geography = geo_locations[geo_country_fields].drop_duplicates().reset_index(drop=True)
require_unique(geography, ['iso3'], 'CEPII country attributes')

display(pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'years': f'{df.year.min()}–{df.year.max()}' if 'year' in df else 'statisch'}
    for name, df in [('nmc', nmc), ('national_trade', national_trade), ('dyadic_trade', dyadic_trade),
                     ('pwt', pwt), ('maddison', maddison), ('distances', distances)]
]))


## 4. Transparente Länderzuordnung

COW-Nummern sind **keine ISO-Nummern**. Die explizite Zuordnung unten kann überprüft und erweitert werden. Historische Einheiten erhalten eigene Kennungen (`SUN`, `DDR`, `CSK`, `YUG`, `SCG`, `YAR`, `YPR`). Westdeutschland wird als `FRG` bewahrt und nicht automatisch mit der PWT-Reihe `DEU` verbunden.

Die Regeln sind bewusst konservativ: Übergangsjahre und bekannte unterschiedliche Gebietsstände werden für quellenübergreifende Haupttabellen markiert. Für Belgien/Luxemburg werden Handelsjahre vor 1997 wegen gemeinsamer Handelsstatistik separat markiert. Für Tansania wird die unterschiedliche Abdeckung der PWT-Mainland-Reihe gekennzeichnet. Alle ursprünglichen Tabellen bleiben dennoch verfügbar.


In [ ]:
COW_TO_ISO = {
 2:'USA',20:'CAN',31:'BHS',40:'CUB',41:'HTI',42:'DOM',51:'JAM',52:'TTO',53:'BRB',54:'DMA',55:'GRD',
 56:'LCA',57:'VCT',58:'ATG',60:'KNA',70:'MEX',80:'BLZ',90:'GTM',91:'HND',92:'SLV',93:'NIC',94:'CRI',95:'PAN',
 100:'COL',101:'VEN',110:'GUY',115:'SUR',130:'ECU',135:'PER',140:'BRA',145:'BOL',150:'PRY',155:'CHL',160:'ARG',165:'URY',
 200:'GBR',205:'IRL',210:'NLD',211:'BEL',212:'LUX',220:'FRA',221:'MCO',223:'LIE',225:'CHE',230:'ESP',232:'AND',235:'PRT',
 255:'DEU',260:'FRG',265:'DDR',290:'POL',300:'AUT_HUN',305:'AUT',310:'HUN',315:'CSK',316:'CZE',317:'SVK',325:'ITA',
 331:'SMR',338:'MLT',339:'ALB',341:'MNE',343:'MKD',344:'HRV',345:'SRB',346:'BIH',347:'XKX',349:'SVN',350:'GRC',
 352:'CYP',355:'BGR',359:'MDA',360:'ROU',365:'RUS',366:'EST',367:'LVA',368:'LTU',369:'UKR',370:'BLR',371:'ARM',
 372:'GEO',373:'AZE',375:'FIN',380:'SWE',385:'NOR',390:'DNK',395:'ISL',402:'CPV',403:'STP',404:'GNB',411:'GNQ',
 420:'GMB',432:'MLI',433:'SEN',434:'BEN',435:'MRT',436:'NER',437:'CIV',438:'GIN',439:'BFA',450:'LBR',451:'SLE',452:'GHA',
 461:'TGO',471:'CMR',475:'NGA',481:'GAB',482:'CAF',483:'TCD',484:'COG',490:'COD',500:'UGA',501:'KEN',510:'TZA',
 511:'ZAN',516:'BDI',517:'RWA',520:'SOM',522:'DJI',530:'ETH',531:'ERI',540:'AGO',541:'MOZ',551:'ZMB',552:'ZWE',
 553:'MWI',560:'ZAF',565:'NAM',570:'LSO',571:'BWA',572:'SWZ',580:'MDG',581:'COM',590:'MUS',591:'SYC',600:'MAR',
 615:'DZA',616:'TUN',620:'LBY',625:'SDN',626:'SSD',630:'IRN',640:'TUR',645:'IRQ',651:'EGY',652:'SYR',660:'LBN',
 663:'JOR',666:'ISR',670:'SAU',678:'YAR',679:'YEM',680:'YPR',690:'KWT',692:'BHR',694:'QAT',696:'ARE',698:'OMN',
 700:'AFG',701:'TKM',702:'TJK',703:'KGZ',704:'UZB',705:'KAZ',710:'CHN',712:'MNG',713:'TWN',730:'KOREA_HIST',
 731:'PRK',732:'KOR',740:'JPN',750:'IND',760:'BTN',770:'PAK',771:'BGD',775:'MMR',780:'LKA',781:'MDV',790:'NPL',
 800:'THA',811:'KHM',812:'LAO',816:'VNM',817:'VNS',820:'MYS',830:'SGP',835:'BRN',840:'PHL',850:'IDN',860:'TLS',
 900:'AUS',910:'PNG',920:'NZL',935:'VUT',940:'SLB',946:'KIR',947:'TUV',950:'FJI',955:'TON',970:'NRU',983:'MHL',
 986:'PLW',987:'FSM',990:'WSM'
}
HISTORICAL_IDS = {'SUN','DDR','FRG','CSK','YUG','SCG','YAR','YPR','VNS','VDR','ZAN','AUT_HUN','KOREA_HIST'}
MODERN_MIN_YEAR = {
    'RUS':1992, 'DEU':1991, 'SRB':2007, 'VNM':1977, 'YEM':1991,
    'CZE':1993, 'SVK':1993, 'PAK':1972, 'ETH':1994, 'SDN':2012, 'MYS':1966,
    'EST':1992, 'LVA':1992, 'LTU':1992, 'UKR':1992, 'BLR':1992, 'ARM':1992,
    'GEO':1992, 'AZE':1992, 'KAZ':1992, 'KGZ':1992, 'TJK':1992, 'TKM':1992,
    'UZB':1992, 'SVN':1993, 'HRV':1993, 'BIH':1993, 'MNE':2007, 'SSD':2012
}

# Mitgliedschaft je Jahr, bei Übergängen kann das Jahr teilweise sein.
membership = pd.concat([
    pd.DataFrame({'ccode': int(r.ccode), 'year': range(int(r.styear), int(r.endyear)+1)})
    for r in state_list.itertuples()
], ignore_index=True).drop_duplicates(['ccode','year'])
membership['in_cow_system'] = True

def add_cow_identity(df):
    df = df.copy()
    df['iso3'] = df['ccode'].map(COW_TO_ISO).astype('string')
    df.loc[(df.ccode == 365) & (df.year.between(1922,1991)), 'iso3'] = 'SUN'
    df.loc[(df.ccode == 365) & (df.year < 1922), 'iso3'] = 'RUS_HIST'
    df.loc[(df.ccode == 345) & (df.year < 1992), 'iso3'] = 'YUG'
    df.loc[(df.ccode == 345) & (df.year.between(1992,2006)), 'iso3'] = 'SCG'
    df.loc[(df.ccode == 816) & (df.year < 1977), 'iso3'] = 'VDR'
    df = df.merge(membership, on=['ccode','year'], how='left', validate='many_to_one')
    df['in_cow_system'] = df['in_cow_system'].astype('boolean').fillna(False).astype(bool)
    return df

def add_scope_flags(df):
    df = df.copy()
    df['scope_review_required'] = df.iso3.isna() | df.iso3.isin(HISTORICAL_IDS | {'RUS_HIST'})
    df['scope_reason'] = np.where(df['scope_review_required'], 'historical_or_unmapped_entity', '')
    for iso, first_year in MODERN_MIN_YEAR.items():
        mask = df.iso3.eq(iso).fillna(False) & df.year.lt(first_year)
        df.loc[mask, 'scope_review_required'] = True
        df.loc[mask, 'scope_reason'] = 'territorial_transition_before_' + str(first_year)
    mask = df.iso3.eq('TZA').fillna(False)
    df.loc[mask, 'scope_review_required'] = True
    df.loc[mask, 'scope_reason'] = 'PWT_mainland_vs_other_sources_review'
    df['trade_scope_review'] = df.iso3.isin(['BEL','LUX']) & df.year.lt(1997)
    return df

nmc = add_scope_flags(add_cow_identity(nmc))
national_trade = add_scope_flags(add_cow_identity(national_trade))
pwt = add_scope_flags(pwt)
maddison = add_scope_flags(maddison)
country_mapping = state_list[['ccode','statenme','styear','endyear']].copy()
country_mapping['default_iso3'] = country_mapping.ccode.map(COW_TO_ISO)
display(country_mapping.head(8))


## 5. SIPRI ins lange Format bringen

Regionsüberschriften werden von echten Länderreihen getrennt. Historische SIPRI-Reihen werden bewahrt. Zeichen wie `...` und `xxx` werden nicht als Null interpretiert. Fußnoten bleiben als Text erhalten. SIPRI-Schätzungen/Unsicherheit sind zusätzlich über Zellfarben im Original markiert; die Farbkennzeichnung wird hier **nicht vollständig in quantitative Qualitätsflags übersetzt**. Für eine spätere Sensitivitätsanalyse Original und Fußnoten konsultieren.


In [ ]:
# Exakte normalisierte Namen aus den gelieferten Tabellen plus explizite Aliase.
name_candidates = {}
for frame, iso_col, name_col in [
    (pwt, 'iso3', 'pwt_country'), (maddison, 'iso3', 'maddison_country'),
    (geo_locations, 'iso3', 'country')]:
    for iso, name in frame[[iso_col,name_col]].drop_duplicates().itertuples(index=False, name=None):
        name_candidates.setdefault(normalize_name(name), set()).add(str(iso))
for row in state_list.itertuples():
    if row.ccode in COW_TO_ISO:
        name_candidates.setdefault(normalize_name(row.statenme), set()).add(COW_TO_ISO[row.ccode])

NAME_ALIASES = {
 'Congo, DR':'COD', 'Congo, Republic':'COG', "Cote d'Ivoire":'CIV', 'Gambia, The':'GMB',
 'Korea, North':'PRK', 'Korea, South':'KOR', 'Timor Leste':'TLS', 'Kyrgyz Republic':'KGZ',
 'Viet Nam':'VNM', 'Czechia':'CZE', 'North Macedonia':'MKD', 'Eswatini':'SWZ', 'Türkiye':'TUR',
 'USSR':'SUN', 'Yugoslavia':'YUG', 'Czechoslovakia':'CSK', 'German Democratic Republic':'DDR',
 'Yemen, North':'YAR', 'Germany':'DEU', 'Serbia':'SRB', 'Russia':'RUS', 'Belgium':'BEL',
 'United States of America':'USA', 'Cape Verde':'CPV', 'Taiwan':'TWN', 'Laos':'LAO', 'Kosovo':'XKX',
 'Romania':'ROU'
}
aliases = {normalize_name(k):v for k,v in NAME_ALIASES.items()}
def resolve_name(name):
    key = normalize_name(name)
    if key in aliases:
        return aliases[key]
    candidates = name_candidates.get(key, set())
    return next(iter(candidates)) if len(candidates) == 1 else pd.NA

sipri_wide = raw['sipri'].copy()
sipri_wide['iso3'] = sipri_wide['Country'].map(resolve_name).astype('string')
year_columns = [c for c in sipri_wide.columns if isinstance(c, (int, np.integer)) and 1900 <= c <= 2100]
value_count = sipri_wide[year_columns].apply(pd.to_numeric, errors='coerce').notna().sum(axis=1)
sipri_aggregate_rows = sipri_wide.loc[sipri_wide.Country.eq('European Union')].copy()
sipri_unmapped_rows = sipri_wide.loc[sipri_wide.iso3.isna() & value_count.gt(0) &
                                    ~sipri_wide.Country.eq('European Union'), ['Country','Notes']].copy()
sipri_excluded_headings = sipri_wide.loc[sipri_wide.iso3.isna() & value_count.eq(0), ['Country']].dropna()
if len(sipri_unmapped_rows):
    print('Nicht zugeordnete SIPRI-Reihen mit Zahlen; manuell prüfen:')
    display(sipri_unmapped_rows)

sipri = sipri_wide.loc[sipri_wide.iso3.notna()].melt(
    id_vars=['Country','iso3','Notes'], value_vars=year_columns,
    var_name='year', value_name='military_spending_raw')
sipri['year'] = sipri['year'].astype(int)
sipri['military_spending_real_musd'] = pd.to_numeric(sipri['military_spending_raw'], errors='coerce')
sipri.loc[sipri.military_spending_real_musd.lt(0), 'military_spending_real_musd'] = np.nan
sipri = sipri.rename(columns={'Country':'sipri_country', 'Notes':'sipri_notes'})
sipri = add_scope_flags(sipri)
require_unique(sipri, ['iso3','year'], 'SIPRI long')

# Vorjahr über Jahreskennung verknüpfen, nicht einfach zur vorherigen Zeile springen.
previous = sipri[['iso3','year','military_spending_real_musd','scope_review_required']].copy()
previous['year'] += 1
previous = previous.rename(columns={'military_spending_real_musd':'spending_previous_year',
                                    'scope_review_required':'previous_scope_review'})
sipri = sipri.merge(previous, on=['iso3','year'], how='left', validate='one_to_one')
valid = sipri.military_spending_real_musd.ge(0) & sipri.spending_previous_year.gt(0)
sipri['military_growth_pct'] = np.nan
sipri.loc[valid, 'military_growth_pct'] = 100 * (
    sipri.loc[valid,'military_spending_real_musd'] / sipri.loc[valid,'spending_previous_year'] - 1)
sipri['growth_outlier_review'] = sipri.military_growth_pct.abs().gt(100)
sipri['military_growth_eligible'] = (~sipri.scope_review_required &
    ~sipri.previous_scope_review.astype('boolean').fillna(True))
display(sipri[['iso3','year','military_spending_real_musd','military_growth_pct']].dropna().head())


## 6. Kriegsdaten und geografische Tabellen

Hier werden die Originaltabellen bereinigt. Die spätere Erweiterung bildet zusätzlich jährliche Beteiligungsmerkmale aus beiden Teilnahmeepisoden und dokumentiert die Grenzen der Kriegsabdeckung ausdrücklich.


In [ ]:
wars = raw['wars'].copy()
date_fields = [c for c in wars.columns if c.startswith(('StartMonth','StartDay','StartYear','EndMonth','EndDay','EndYear'))]
for col in date_fields + ['BatDeath']:
    wars[col] = pd.to_numeric(wars[col], errors='coerce')
    wars.loc[wars[col].lt(0), col] = np.nan
wars['war_id'] = wars['WarNum']
wars['participant_row_id'] = np.arange(len(wars))
for period in [1,2]:
    for prefix in ['Start','End']:
        parts = {part.lower(): wars[f'{prefix}{part}{period}'] for part in ['Year','Month','Day']}
        wars[f'{prefix.lower()}_date_{period}'] = pd.to_datetime(parts, errors='coerce')
wars['duration_days_first_period'] = (wars.end_date_1 - wars.start_date_1).dt.days + 1
wars.loc[wars.duration_days_first_period.lt(1), 'duration_days_first_period'] = np.nan
print(f'Kriegsbeteiligungen: {len(wars):,}; unterschiedliche Kriege: {wars.war_id.nunique()}')
print(f'Geografische Ortszeilen: {len(geo_locations):,}; eindeutige Länder: {len(geography):,}')
print(f'Bilaterale Entfernungen: {len(distances):,}; Distanzmaß: {DISTANCE_COLUMN}')


## 7. Länderjahre und wirtschaftliche Kennzahlen vorbereiten

NMC-Personal und Bevölkerung sind in Tausend angegeben. PWT-BIP ist in Millionen 2021-US-Dollar bei Kaufkraftparität, Erwerbstätige und Bevölkerung ebenfalls in Millionen. `rgdpo / emp` ergibt daher BIP je Erwerbstätigem. Für den Wohlstandsvergleich innerhalb eines Jahres wird `cgdpe / pop` verwendet.

Maddison-Wachstum verwendet exakt die Jahre t und t+5. Fehlende Endpunkte werden nicht interpoliert. Die COW-geführte Tabelle ist der gemeinsame Arbeitsrahmen für die ausgewählten Hypothesen; die vollständigen PWT-/Maddison-/SIPRI-Tabellen bleiben separat verfügbar.


In [ ]:
nmc['military_per_1000'] = np.where(nmc.tpop.gt(0), 1000 * nmc.milper / nmc.tpop, np.nan)
pwt['productivity'] = np.where(pwt.emp.gt(0) & pwt.rgdpo.gt(0), pwt.rgdpo / pwt.emp, np.nan)
pwt['gdp_pc_comparable'] = np.where(pwt['pop'].gt(0) & pwt.cgdpe.gt(0), pwt.cgdpe / pwt['pop'], np.nan)

future = maddison[['iso3','year','gdppc']].rename(columns={'gdppc':'gdppc_t_plus_5'}).copy()
future['year'] -= 5
maddison = maddison.merge(future, on=['iso3','year'], how='left', validate='one_to_one')
valid = maddison.gdppc.gt(0) & maddison.gdppc_t_plus_5.gt(0)
maddison['growth_next_5y_pct'] = np.nan
maddison.loc[valid,'growth_next_5y_pct'] = 100 * (
    (maddison.loc[valid,'gdppc_t_plus_5'] / maddison.loc[valid,'gdppc']) ** (1/5) - 1)

country_year_keys = pd.concat([nmc[['ccode','year']], national_trade[['ccode','year']]], ignore_index=True).drop_duplicates()
country_year = add_scope_flags(add_cow_identity(country_year_keys))
country_year = country_year.merge(nmc[['ccode','year','milper','tpop','military_per_1000']],
                                on=['ccode','year'], how='left', validate='one_to_one')
country_year = country_year.merge(national_trade[['ccode','year','imports','exports']],
                                on=['ccode','year'], how='left', validate='one_to_one')
country_year = country_year.merge(pwt[['iso3','year','pwt_country','productivity','gdp_pc_comparable']],
                                on=['iso3','year'], how='left', validate='many_to_one')
country_year = country_year.merge(maddison[['iso3','year','gdppc','gdppc_t_plus_5','growth_next_5y_pct']],
                                on=['iso3','year'], how='left', validate='many_to_one')
country_year = country_year.merge(sipri[['iso3','year','military_spending_real_musd','military_growth_pct',
                                       'growth_outlier_review','military_growth_eligible','sipri_notes']],
                                on=['iso3','year'], how='left', validate='many_to_one')
country_year['analysis_period'] = country_year.year.between(START_YEAR, END_YEAR)
country_year['cross_source_eligible'] = (~country_year.scope_review_required) & country_year.in_cow_system
require_unique(country_year, ['ccode','year'], 'Country-year master')
scope_review = country_year.loc[country_year.analysis_period & country_year.scope_review_required,
                                 ['ccode','iso3','year','scope_reason']].copy()
display(country_year.loc[country_year.analysis_period].head())


## 8. Gerichtete Handelsströme und Abdeckung

`flow1` sind Importe von Land 1 aus Land 2, `flow2` die umgekehrten Importe. **Kein unbekannter Wert wird zu Null.** Alle bekannten Partnerströme zählen in den Abdeckungssummen mit, auch wenn ein Partner nicht in PWT vorhanden ist.

Die nationale Import-/Exportsumme und die Summe bilateraler Ströme können sich wegen Berichtsmethoden und Abdeckung unterscheiden. Die 90–110%-Regel ist eine transparente Screening-Annahme und muss später variiert werden. Sie beweist keine vollständige Partnererfassung. `import_hhi_observed` beschreibt die Konzentration der **beobachteten** Partnerstruktur, keine garantiert vollständige Weltstruktur.


In [ ]:
base = dyadic_trade.loc[dyadic_trade.year.between(START_YEAR,END_YEAR)].copy()
part_a = base[['ccode1','ccode2','year','flow1']].rename(columns={
    'ccode1':'importer_ccode','ccode2':'exporter_ccode','flow1':'trade_value_musd'})
part_b = base[['ccode2','ccode1','year','flow2']].rename(columns={
    'ccode2':'importer_ccode','ccode1':'exporter_ccode','flow2':'trade_value_musd'})
directed_trade = pd.concat([part_a,part_b], ignore_index=True)
directed_trade = directed_trade.loc[directed_trade.importer_ccode.ne(directed_trade.exporter_ccode)].copy()
require_unique(directed_trade, ['importer_ccode','exporter_ccode','year'], 'Directed trade')

identity_cols = ['ccode','year','iso3','scope_review_required','trade_scope_review','in_cow_system']
for role in ['importer','exporter']:
    identities = country_year[identity_cols].rename(columns={
        'ccode': role+'_ccode', 'iso3':role+'_iso3',
        'scope_review_required':role+'_scope_review', 'trade_scope_review':role+'_trade_review',
        'in_cow_system':role+'_in_cow_system'})
    directed_trade = directed_trade.merge(identities, on=[role+'_ccode','year'], how='left', validate='many_to_one')

trade_totals = national_trade[['ccode','year','imports','exports']]
directed_trade = directed_trade.merge(
    trade_totals[['ccode','year','imports']].rename(columns={'ccode':'importer_ccode','imports':'national_imports_musd'}),
    on=['importer_ccode','year'], how='left', validate='many_to_one')
directed_trade['partner_import_share'] = np.where(
    directed_trade.national_imports_musd.gt(0),
    directed_trade.trade_value_musd / directed_trade.national_imports_musd, np.nan)
directed_trade.loc[~directed_trade.partner_import_share.between(0,1), 'partner_import_share'] = np.nan

import_summary = directed_trade.groupby(['importer_ccode','year'], as_index=False).agg(
    observed_imports_musd=('trade_value_musd',lambda x:x.sum(min_count=1)),
    known_partner_flows=('trade_value_musd','count'),
    listed_partner_flows=('trade_value_musd','size'))
import_summary = import_summary.rename(columns={'importer_ccode':'ccode'})
import_summary = import_summary.merge(trade_totals[['ccode','year','imports']], on=['ccode','year'], validate='one_to_one')
import_summary['import_coverage_ratio'] = np.where(import_summary.imports.gt(0),
    import_summary.observed_imports_musd/import_summary.imports,np.nan)
import_summary['import_coverage_ok'] = import_summary.import_coverage_ratio.between(COVERAGE_MIN,COVERAGE_MAX)

weights = directed_trade[['importer_ccode','year','trade_value_musd']].merge(
    import_summary[['ccode','year','observed_imports_musd']].rename(columns={'ccode':'importer_ccode'}),
    on=['importer_ccode','year'],validate='many_to_one')
weights['share_sq'] = np.where(weights.observed_imports_musd.gt(0),
    (weights.trade_value_musd / weights.observed_imports_musd)**2,np.nan)
hhi = weights.groupby(['importer_ccode','year'])['share_sq'].sum(min_count=1).rename('import_hhi_observed').reset_index()
import_summary = import_summary.merge(hhi.rename(columns={'importer_ccode':'ccode'}),on=['ccode','year'],validate='one_to_one')
country_year = country_year.merge(import_summary.drop(columns='imports'),on=['ccode','year'],how='left',validate='one_to_one')
print('Gerichtete Handelszeilen:', len(directed_trade))
display(import_summary[['import_coverage_ratio','import_hhi_observed','known_partner_flows']].describe())
del weights, part_a, part_b


## 9. Staatenpaare für H3 und H4

Jedes ungeordnete Staatenpaar wird pro Jahr nur einmal gespeichert. Der Handelsanteil ist das Mittel der beiden gerichteten Importanteile an den jeweiligen nationalen Importen. Der Aufrüstungsunterschied ist der **absolute Unterschied** zweier jährlicher Wachstumsraten in Prozentpunkten. Fehlende Vorjahre werden nicht überbrückt.

Die Wohlstandslücke ist die absolute Differenz logarithmierter vergleichbarer BIP-pro-Kopf-Werte. `distance_km` steht als Kontrollvariable bereit; eine bloße bivariate Korrelation prüft nicht, ob Wohlstandsnähe wichtiger als geografische Nähe ist.


In [ ]:
dyad_year = base[['ccode1','ccode2','year','flow1','flow2']].copy()
assert dyad_year.ccode1.lt(dyad_year.ccode2).all(), 'COW-Paarordnung prüfen.'
attributes = ['ccode','year','iso3','cross_source_eligible','trade_scope_review','imports',
              'gdp_pc_comparable','military_growth_pct','growth_outlier_review','military_growth_eligible']
for side in ['1','2']:
    renamed = country_year[attributes].rename(columns={c:c+side for c in attributes if c!='year'})
    dyad_year = dyad_year.merge(renamed,on=['ccode'+side,'year'],how='left',validate='many_to_one')
valid_shares = (dyad_year.imports1.gt(0)&dyad_year.imports2.gt(0)&
                dyad_year.flow1.ge(0)&dyad_year.flow2.ge(0))
share1 = dyad_year.flow1/dyad_year.imports1
share2 = dyad_year.flow2/dyad_year.imports2
valid_shares &= share1.between(0,1)&share2.between(0,1)
dyad_year['mutual_import_share'] = np.where(valid_shares,(share1+share2)/2,np.nan)
dyad_year['military_growth_gap_pp'] = (dyad_year.military_growth_pct1-dyad_year.military_growth_pct2).abs()
positive_gdp = dyad_year.gdp_pc_comparable1.gt(0)&dyad_year.gdp_pc_comparable2.gt(0)
dyad_year['wealth_gap_log'] = np.nan
dyad_year.loc[positive_gdp,'wealth_gap_log'] = (
    np.log(dyad_year.loc[positive_gdp,'gdp_pc_comparable1'])-
    np.log(dyad_year.loc[positive_gdp,'gdp_pc_comparable2'])).abs()
dyad_year = dyad_year.merge(
    distances[['iso3_origin','iso3_destination',DISTANCE_COLUMN]].rename(columns={
        'iso3_origin':'iso31','iso3_destination':'iso32',DISTANCE_COLUMN:'distance_km'}),
    on=['iso31','iso32'],how='left',validate='many_to_one')
dyad_year['pair_eligible'] = (
    dyad_year.cross_source_eligible1.astype('boolean').fillna(False)&dyad_year.cross_source_eligible2.astype('boolean').fillna(False)&
    ~dyad_year.trade_scope_review1.astype('boolean').fillna(True)&~dyad_year.trade_scope_review2.astype('boolean').fillna(True))
require_unique(dyad_year,['ccode1','ccode2','year'],'Dyad-year master')
display(dyad_year[['iso31','iso32','year','mutual_import_share','military_growth_gap_pp','wealth_gap_log','distance_km']].head())
del base


## 10. Exportentfernung für H8

Exporte von A nach B entsprechen den erfassten Importen von B aus A. Der Exporteur ist deshalb `exporter_ccode`. CEPII-Distanzen sind statische geografische Kennzahlen, keine historischen Transportwege oder Jahresdaten. Die Zuordnung insbesondere historischer Gebietsstände bleibt eine Annahme.

Die gewichtete Entfernung wird zunächst nur über bekannte Ströme mit bekannter Distanz berechnet und ausdrücklich als `export_distance_observed_km` bezeichnet. Die Haupttabelle verwendet sie nur bei ausreichender nationaler Handelsabdeckung und mindestens 95% Distanzabdeckung der bekannten Exportwerte. Eine strengere 100%-Prüfung bleibt später möglich.


In [ ]:
directed_trade = directed_trade.merge(
    distances[['iso3_origin','iso3_destination',DISTANCE_COLUMN]].rename(columns={
        'iso3_origin':'exporter_iso3','iso3_destination':'importer_iso3',DISTANCE_COLUMN:'distance_km'}),
    on=['exporter_iso3','importer_iso3'],how='left',validate='many_to_one')
# Historische/zu prüfende Partner werden nicht mit heutiger Geografie gleichgesetzt.
distance_eligible = (
    directed_trade.distance_km.gt(0)&
    ~directed_trade.exporter_scope_review.astype('boolean').fillna(True)&
    ~directed_trade.importer_scope_review.astype('boolean').fillna(True)&
    ~directed_trade.exporter_trade_review.astype('boolean').fillna(True)&
    ~directed_trade.importer_trade_review.astype('boolean').fillna(True))
directed_trade['distance_trade_value'] = directed_trade.trade_value_musd.where(distance_eligible)
directed_trade['weighted_distance_value'] = directed_trade.distance_trade_value*directed_trade.distance_km
export_summary = directed_trade.groupby(['exporter_ccode','year'],as_index=False).agg(
    observed_exports_musd=('trade_value_musd',lambda x:x.sum(min_count=1)),
    exports_with_distance_musd=('distance_trade_value',lambda x:x.sum(min_count=1)),
    weighted_distance_sum=('weighted_distance_value',lambda x:x.sum(min_count=1)))
export_summary = export_summary.rename(columns={'exporter_ccode':'ccode'})
export_summary = export_summary.merge(trade_totals[['ccode','year','exports']],on=['ccode','year'],validate='one_to_one')
export_summary['export_coverage_ratio'] = np.where(export_summary.exports.gt(0),
    export_summary.observed_exports_musd/export_summary.exports,np.nan)
export_summary['distance_coverage_ratio'] = np.where(export_summary.observed_exports_musd.gt(0),
    export_summary.exports_with_distance_musd/export_summary.observed_exports_musd,np.nan)
export_summary['export_distance_observed_km'] = np.where(export_summary.exports_with_distance_musd.gt(0),
    export_summary.weighted_distance_sum/export_summary.exports_with_distance_musd,np.nan)
export_summary['export_coverage_ok'] = export_summary.export_coverage_ratio.between(COVERAGE_MIN,COVERAGE_MAX)
export_summary['distance_coverage_ok'] = export_summary.distance_coverage_ratio.ge(DISTANCE_COVERAGE_MIN)
country_year = country_year.merge(export_summary.drop(columns=['exports','weighted_distance_sum']),
                                on=['ccode','year'],how='left',validate='one_to_one')
display(export_summary[['export_coverage_ratio','distance_coverage_ratio','export_distance_observed_km']].describe())


## 11. Fertige Hypothesentabellen und Ausschlussprotokolle

Die Tabellen `h1`, `h2`, `h3`, `h4`, `h8` enthalten nur für die jeweilige Hypothese vollständige Wertepaare. Es gibt **keinen globalen dropna-Schritt** über alle Quellen. `audit` zählt den Verlust pro Regel in fester Reihenfolge. Die detaillierten Prüfflags bleiben in `country_year` und `dyad_year` zugänglich.

H2/H8 verwenden zusätzlich Handelsabdeckung. Bei H3/H4 werden echte Null-Handelsanteile erhalten; sie bedeuten keine unbekannten Werte. Extreme militärische Wachstumsraten bleiben erhalten und werden nur gekennzeichnet.


In [ ]:
audit_rows = []
def prepare_pairs(df, hypothesis, x, y, keys, rules):
    current = df.copy()
    audit_rows.append({'hypothesis':hypothesis,'step':'start','remaining':len(current),'removed':0})
    for label, rule in rules:
        mask = rule(current).astype('boolean').fillna(False)
        removed = len(current)-int(mask.sum())
        current = current.loc[mask].copy()
        audit_rows.append({'hypothesis':hypothesis,'step':label,'remaining':len(current),'removed':removed})
    finite = np.isfinite(pd.to_numeric(current[x],errors='coerce')) & np.isfinite(pd.to_numeric(current[y],errors='coerce'))
    removed = len(current)-int(finite.sum())
    current = current.loc[finite].copy()
    audit_rows.append({'hypothesis':hypothesis,'step':'finite_complete_pairs','remaining':len(current),'removed':removed})
    require_unique(current,keys,hypothesis)
    current.attrs.update({'hypothesis':hypothesis,'x':x,'y':y})
    return current

country_rules = [
    ('analysis_period',lambda d:d.analysis_period),
    ('country_scope_and_membership',lambda d:d.cross_source_eligible)]
trade_country_rules = country_rules + [('trade_scope',lambda d:~d.trade_scope_review)]
h1 = prepare_pairs(country_year,'H1','productivity','military_per_1000',['ccode','year'],country_rules)
h2 = prepare_pairs(country_year,'H2','import_hhi_observed','military_per_1000',['ccode','year'],
    trade_country_rules+[('import_coverage',lambda d:d.import_coverage_ok)])
pair_rules = [('pair_scope',lambda d:d.pair_eligible)]
h3 = prepare_pairs(dyad_year,'H3','mutual_import_share','military_growth_gap_pp',
                   ['ccode1','ccode2','year'],pair_rules+[
                       ('comparable_previous_years', lambda d:
                           d.military_growth_eligible1.astype('boolean').fillna(False)&
                           d.military_growth_eligible2.astype('boolean').fillna(False))])
h4 = prepare_pairs(dyad_year,'H4','wealth_gap_log','mutual_import_share',
                   ['ccode1','ccode2','year'],pair_rules)
h8 = prepare_pairs(country_year,'H8','export_distance_observed_km','growth_next_5y_pct',['ccode','year'],
    trade_country_rules+[
        ('export_coverage',lambda d:d.export_coverage_ok),
        ('distance_coverage',lambda d:d.distance_coverage_ok)])

# Explorative Kandidaten behalten alle Qualitätsflags. Nicht mit der qualitätsgefilterten
# Haupttabelle h8 verwechseln; zusätzliche Zeilen sind keine automatisch zulässigen Fälle.
h8_candidate_mask = (country_year.analysis_period & country_year.cross_source_eligible &
    ~country_year.trade_scope_review & country_year.export_distance_observed_km.gt(0) &
    np.isfinite(country_year.growth_next_5y_pct))
h8_candidates = country_year.loc[h8_candidate_mask].copy()
h8_coverage_sensitivity = pd.DataFrame([
    {'national_coverage_min':low,'national_coverage_max':high,
     'distance_coverage_min':dist_min,
     'n_complete_pairs':int((h8_candidates.export_coverage_ratio.between(low,high)&
                             h8_candidates.distance_coverage_ratio.ge(dist_min)).sum())}
    for low,high in [(0.95,1.05),(COVERAGE_MIN,COVERAGE_MAX),(0.80,1.20)]
    for dist_min in [1.0,0.95,0.90]
])

analysis_tables = {'H1':h1,'H2':h2,'H3':h3,'H4':h4,'H8':h8}
audit = pd.DataFrame(audit_rows)
sample_summary = pd.DataFrame([
    {'hypothesis':name,'n_complete_pairs':len(df),'minimum_5000_met':len(df)>=MIN_PAIRS,
     'first_year':df.year.min() if len(df) else pd.NA,'last_year':df.year.max() if len(df) else pd.NA,
     'unit':'country-year' if name in ['H1','H2','H8'] else 'unordered-dyad-year',
     'x':df.attrs['x'],'y':df.attrs['y']}
    for name,df in analysis_tables.items()])
display(sample_summary)
display(audit)
display(h8_coverage_sensitivity)
if not sample_summary.minimum_5000_met.all():
    print('ACHTUNG: Nicht jede Hypothese erreicht 5.000 Paare. Prüfe Datenabdeckung/Zeitraum oder passe die Hypothese an. Keine Werte künstlich vervielfachen.')


## 12. Qualitätskontrolle und Datenwörterbuch

Die Kontrollen prüfen Schlüssel, Kennzahlengrenzen und vorhandene Daten. Sie garantieren weder historische Vergleichbarkeit noch Unabhängigkeit der Beobachtungen. Fehlende Werte bleiben sichtbar.


In [ ]:
assert len(raw)==10
assert h1.productivity.gt(0).all()
assert h1.military_per_1000.ge(0).all()
assert h2.import_hhi_observed.between(0,1+1e-9).all()
assert h3.mutual_import_share.between(0,1).all()
assert h3.military_growth_gap_pp.ge(0).all()
assert h4.wealth_gap_log.ge(0).all()
assert h8.export_distance_observed_km.gt(0).all()
assert h8.distance_coverage_ratio.ge(DISTANCE_COVERAGE_MIN).all()
assert h8.growth_next_5y_pct.gt(-100).all()

unmapped_cow_codes = pd.concat([
    nmc.loc[nmc.iso3.isna(),['ccode','stateabb','year']],
    national_trade.loc[national_trade.iso3.isna(),['ccode','stateabb','year']]
]).drop_duplicates()
data_dictionary = pd.DataFrame([
    ('productivity','real GDP / persons engaged','2021 PPP USD per worker','PWT rgdpo / emp'),
    ('military_per_1000','military personnel / population * 1000','persons per 1000 inhabitants','NMC milper / tpop'),
    ('import_hhi_observed','sum of squared observed import shares','0 to 1','COW bilateral imports'),
    ('mutual_import_share','mean of directed import shares','0 to 1','COW bilateral / national imports'),
    ('military_growth_pct','real expenditure change from exact previous year','percent','SIPRI constant 2024 USD'),
    ('military_growth_gap_pp','absolute difference of annual growth rates','percentage points','SIPRI'),
    ('gdp_pc_comparable','expenditure-side GDP / population at current PPPs','2021 PPP USD per person','PWT cgdpe / pop'),
    ('wealth_gap_log','absolute log GDP-per-capita difference','log units','PWT'),
    ('export_distance_observed_km','export-value-weighted distance among known-distance flows','km','COW + CEPII distw'),
    ('growth_next_5y_pct','compound annual GDP-per-capita growth from t to t+5','percent per year','Maddison gdppc'),
    ('import_coverage_ratio','sum of observed bilateral imports / national imports','ratio','COW'),
    ('export_coverage_ratio','sum of observed bilateral exports / national exports','ratio','COW'),
    ('distance_coverage_ratio','known exports with eligible distance / known exports','ratio','COW + CEPII'),
],columns=['variable','definition','unit','source'])
display(data_dictionary)
print('Historisch/nicht zugeordnet zu prüfende Länderjahre im Analysezeitraum:',len(scope_review))
print('Nicht zugeordnete COW-Zeilen aller Jahre:',len(unmapped_cow_codes))
display(scope_review.groupby(['iso3','scope_reason'],dropna=False).size().rename('rows').reset_index().head(20))
print('Alle strukturellen Kontrollen bestanden. Inhaltliche Qualitätsprüfung bleibt erforderlich.')


## 13. Optionaler Export und nächste Analyseschritte

Standardmäßig wird **nichts gespeichert**. Setze oben `EXPORT_PREPARED = True`, wenn du die Tabellen als CSV in `Prepared/` schreiben möchtest. Ein erneuter Lauf überschreibt nur die dortigen Aufbereitungsdateien.

Für die nächste Analyse stehen zur Verfügung:

| Objekt | Inhalt |
|---|---|
| `raw` | Alle zehn unveränderten importierten Datentabellen |
| `metadata` | PWT-Legende, Maddison-Quellen und SIPRI-Fußnoten |
| `nmc`, `pwt`, `maddison`, `sipri`, `national_trade`, `dyadic_trade` | Bereinigte Quelltabellen, größtenteils mit voller Zeitabdeckung |
| `geo_locations`, `geography`, `distances`, `state_list`, `wars` | Geografie, Länderinformationen und Kriegsbeteiligungen |
| `country_year` | Zusammengeführte Länderjahre mit Qualitätsflags |
| `directed_trade` | Gerichtete Handelsströme und Distanzzuordnung im Analysezeitraum |
| `dyad_year` | Ungeordnete Staatenpaarjahre mit H3-/H4-Variablen |
| `h1`, `h2`, `h3`, `h4`, `h8` / `analysis_tables` | Vollständige Wertepaare je Hypothese |
| `h8_candidates`, `h8_coverage_sensitivity` | Explorative H8-Kandidaten mit Prüfflags; Stichprobengrößen unter verschiedenen Qualitätsregeln |
| `sample_summary`, `audit`, `scope_review`, `sipri_unmapped_rows` | Stichprobengrößen und Prüftabellen |

**Danach:** Verteilungen und Ausreißer visualisieren, n je Hypothese prüfen, Pearson berechnen, Spearman ergänzen, Abdeckungsregeln variieren. Länder-/Jahreseffekte und abhängige Staatenpaar-Beobachtungen berücksichtigen. Die überlappenden Wachstumsfenster von H8 sind nicht unabhängig. Keine automatische Kausalitäts- oder Signifikanzentscheidung aus konventionellen p-Werten treffen.

Die Kriegsdaten sind optional für spätere zusätzliche Untersuchungen. Das Notebook hat sie nicht stillschweigend in die fünf ausgewählten Hypothesen eingebaut.


In [ ]:
if EXPORT_PREPARED:
    export_dir = PROJECT_DIR/'Prepared'
    export_dir.mkdir(exist_ok=True)
    for name,table in analysis_tables.items():
        table.to_csv(export_dir/f'{name.lower()}_prepared.csv',index=False)
    for name,table in {'country_year':country_year,'sample_summary':sample_summary,'audit':audit,
                       'source_catalog':source_catalog,'data_dictionary':data_dictionary,
                       'country_mapping':country_mapping,'scope_review':scope_review,
                       'sipri_unmapped_rows':sipri_unmapped_rows,
                       'h8_coverage_sensitivity':h8_coverage_sensitivity}.items():
        table.to_csv(export_dir/f'{name}.csv',index=False)
    print('Tabellen gespeichert unter',export_dir.resolve())
else:
    print('Datenaufbereitung abgeschlossen. Alle Tabellen liegen im Arbeitsspeicher bereit.')
    print('Optionaler CSV-Export ist deaktiviert.')


## Erweiterte Analyse: Bedienung und Interpretation

Dieser Teil kann auch hinter die vorhandene Aufbereitung kopiert werden.
Das vollständige Notebook oben enthält dieselbe Aufbereitung und läuft allein
mit dem bestehenden Ordner `Data/`. Alle Analysen sind explorativ.

Ausreißer werden markiert und nachvollziehbar verglichen, nicht automatisch
als Fehler gelöscht. Pearson und Spearman werden nebeneinander berichtet.
Standard-p-Werte sind bei abhängigen Länder-/Paarjahresdaten eingeschränkt.
Teilgruppen müssen nicht jeweils die 5.000-Paare-Vorgabe der Hauptanalyse erfüllen.

Am Ende entsteht **eine Excel-Datei inklusive eingebetteter Grafiken** in
`Ergebnisse/Analysebericht_<Berliner Zeitstempel>.xlsx`. Originaldateien werden
nicht verändert. Die Datei ist ein statischer Bericht; Änderungen an Zellen
berechnen die Python-Analysen nicht neu. Dafür das Notebook erneut ausführen.


In [ ]:
# Einmalig bei fehlenden Paketen in einer eigenen Jupyter-Zelle ausführen:
# %pip install pandas numpy scipy matplotlib openpyxl xlrd pillow
import sys
import platform
import warnings
import math
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.drawing.image import Image as ExcelImage
from openpyxl.utils import get_column_letter

CONFIG = {
    'trim_lower': 0.01, 'trim_upper': 0.99,
    'iqr_multiplier': 1.5, 'mad_z_limit': 3.5,
    'min_group_n': 30, 'min_country_years': 10,
    'top_observations': 25, 'bootstrap_repeats': 2000,
    'random_seed': 42, 'coverage_low': 0.90, 'coverage_high': 1.10,
    'sensitivity_delta_r': 0.10,
}
ANALYSIS_CODE_SHA256 = 'c80c9f9efe264c860fe0755b7aa017b0dd798e2277b5c1e7c8385ff79ad253e5'
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
report_tables, chart_items, notices = {}, [], []
run_started = datetime.now(ZoneInfo('Europe/Berlin'))

def complete(df, columns):
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result.loc[np.isfinite(result[columns].to_numpy(dtype=float)).all(axis=1)].copy()

def corr_metrics(df, x, y, standard_p=True):
    d = complete(df, [x, y])
    result = {'n': len(d), 'pearson_r': np.nan, 'p_standard': np.nan,
              'spearman_rho': np.nan, 'status': 'too_few_or_constant'}
    if len(d) >= 3 and d[x].nunique() > 1 and d[y].nunique() > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r, p = stats.pearsonr(d[x], d[y])
            rho, _ = stats.spearmanr(d[x], d[y])
        result.update(pearson_r=float(r), spearman_rho=float(rho),
                      p_standard=float(p) if standard_p else np.nan, status='computed')
    return result

def pair_coverage(df):
    result = df.copy()
    for side in [1, 2]:
        col = f'import_coverage_{side}'
        if col in result:
            result = result.drop(columns=col)
        c = country_year[['ccode', 'year', 'import_coverage_ratio']].rename(
            columns={'ccode': f'ccode{side}', 'import_coverage_ratio': col})
        result = result.merge(c, on=[f'ccode{side}', 'year'], how='left', validate='many_to_one')
    return result

h3_all_coverage, h4_all_coverage = pair_coverage(h3), pair_coverage(h4)
def coverage_mask(d, low=0.90, high=1.10):
    return d.import_coverage_1.between(low, high) & d.import_coverage_2.between(low, high)

h3_analysis = h3_all_coverage.loc[coverage_mask(h3_all_coverage)].copy()
h4_analysis = h4_all_coverage.loc[coverage_mask(h4_all_coverage)].copy()
datasets = {
    'H1': (h1.copy(), 'productivity', 'military_per_1000', 'negative'),
    'H2': (h2.copy(), 'import_hhi_observed', 'military_per_1000', 'positive'),
    'H3': (h3_analysis, 'mutual_import_share', 'military_growth_gap_pp', 'negative'),
    'H4': (h4_analysis, 'wealth_gap_log', 'mutual_import_share', 'negative'),
    'H8': (h8.copy(), 'export_distance_observed_km', 'growth_next_5y_pct', 'positive'),
}
LABELS = {
    'productivity': 'Real GDP per worker (2021 PPP USD)',
    'military_per_1000': 'Military personnel per 1,000 inhabitants',
    'import_hhi_observed': 'Observed import concentration (HHI)',
    'mutual_import_share': 'Mutual import share (0-1)',
    'military_growth_gap_pp': 'Military expenditure growth gap (pp)',
    'wealth_gap_log': 'Absolute log GDP-per-capita gap',
    'export_distance_observed_km': 'Export-weighted partner distance (km)',
    'growth_next_5y_pct': 'Subsequent 5-year annual GDPpc growth (%)',
    'total_gdp_gap_log': 'Absolute log total GDP gap',
    'asymmetry_abs': 'Absolute difference in import shares (0-1)',
}
baseline_rows, counts_rows = [], []
for name, (d, x, y, expected) in datasets.items():
    d = complete(d, [x, y]).reset_index(drop=True)
    keys = ['ccode', 'year'] if 'ccode' in d else ['ccode1', 'ccode2', 'year']
    require_unique(d, keys, name + ' extended sample')
    if 'ccode' not in d:
        d['pair_id'] = d.ccode1.astype(str) + '_' + d.ccode2.astype(str)
    datasets[name] = (d, x, y, expected)
    row = {'hypothesis': name, 'role': 'exploratory' if name == 'H8' else 'original',
           'x': x, 'y': y, 'expected_direction': expected, **corr_metrics(d, x, y)}
    row['minimum_5000_met'] = row['n'] >= 5000
    row['countries'] = d.ccode.nunique() if 'ccode' in d else pd.unique(d[['ccode1','ccode2']].to_numpy().ravel()).size
    row['entities'] = d.ccode.nunique() if 'ccode' in d else d.pair_id.nunique()
    row['years'] = d.year.nunique()
    row['observed_direction_matches'] = (row['pearson_r'] < 0) == (expected == 'negative')
    baseline_rows.append(row)
baseline = pd.DataFrame(baseline_rows)
report_tables['Correlations'] = baseline
report_tables['Coverage_correction'] = pd.DataFrame([
    {'hypothesis': k, 'before': len(orig), 'after': len(datasets[k][0]),
     'removed': len(orig)-len(datasets[k][0]), 'rule': 'Both import coverage ratios in [0.90,1.10]'}
    for k, orig in [('H3', h3), ('H4', h4)]])
display(baseline[['hypothesis','n','pearson_r','spearman_rho','p_standard']])


### Ausreißer und Einfluss einzelner Beobachtungen

Verglichen werden vollständige Daten, IQR-Grenzen, robuster MAD-z-Wert,
1./99.-Perzentil-Trimmen und Winsorisieren. Alle Grenzen werden je Hypothese
auf der vollständigen Ausgangsstichprobe bestimmt. Globales Trimmen kann
echte Struktur entfernen und ist deshalb ausschließlich eine Sensitivität.
Cook's D bezieht sich auf eine einfache lineare Regression mit Konstante.
Die Löschdiagnose berechnet exakt die Änderung von Pearson r beim Weglassen
einer Zeile. Länder- und Jahresdiagnosen folgen zusätzlich, weil einzelne
Länderjahre in vielen Paaren vorkommen können.


In [ ]:
def r_from_moments(a):
    # [n, sum(x), sum(y), sum(x*x), sum(y*y), sum(x*y)], global zentriert.
    a = np.asarray(a, dtype=float)
    n, sx, sy, sxx, syy, sxy = np.moveaxis(a, -1, 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        vx, vy = sxx - sx*sx/n, syy - sy*sy/n
        r = (sxy - sx*sy/n) / np.sqrt(vx*vy)
    return np.where((n >= 3) & (vx > 1e-14) & (vy > 1e-14), np.clip(r, -1, 1), np.nan)

def moment_matrix(d, x, y):
    xx = d[x].to_numpy(float); yy = d[y].to_numpy(float)
    xx = xx - xx.mean(); yy = yy - yy.mean()
    return np.column_stack([np.ones(len(d)), xx, yy, xx*xx, yy*yy, xx*yy])

variant_frames, sensitivity_rows, cutoff_rows, top_rows, outlier_summary = {}, [], [], [], []
for name, (d, x, y, _) in datasets.items():
    flags, thresholds = {}, {}
    for method in ['iqr', 'mad', 'percentile']:
        mark = np.zeros(len(d), dtype=bool)
        for col in [x, y]:
            s = d[col]
            if method == 'iqr':
                q1, q3 = s.quantile([0.25, 0.75]); span = q3-q1
                low, high = q1-CONFIG['iqr_multiplier']*span, q3+CONFIG['iqr_multiplier']*span
                active = span > 0
            elif method == 'mad':
                med = s.median(); mad = (s-med).abs().median()
                radius = CONFIG['mad_z_limit']*mad/0.6744897501960817
                low, high, active = med-radius, med+radius, mad > 0
            else:
                low, high = s.quantile([CONFIG['trim_lower'], CONFIG['trim_upper']]); active = True
            cutoff_rows.append({'hypothesis': name, 'method': method, 'variable': col,
                                'low': low, 'high': high, 'rule_active': active,
                                'note': '' if active else 'Zero IQR/MAD: this variable is not flagged by this rule'})
            if active:
                mark |= ~s.between(low, high).to_numpy()
            if method == 'percentile':
                thresholds[col] = (low, high)
        flags[method] = mark
    winsor = d.copy()
    for col in [x, y]:
        winsor[col] = winsor[col].clip(*thresholds[col])
    variants = {'full': d, 'iqr_excluded': d.loc[~flags['iqr']],
                'mad_excluded': d.loc[~flags['mad']],
                'trim_p01_p99': d.loc[~flags['percentile']], 'winsor_p01_p99': winsor}
    if name == 'H3':
        variants['growth_abs_le_100pct'] = d.loc[d.military_growth_pct1.abs().le(100) & d.military_growth_pct2.abs().le(100)]
    if name == 'H8':
        variants['nonoverlap_start_mod5_0'] = d.loc[d.year.mod(5).eq(0)]
    variant_frames[name] = variants
    base_r = float(baseline.set_index('hypothesis').loc[name, 'pearson_r'])
    for label, v in variants.items():
        m = corr_metrics(v, x, y)
        delta = m['pearson_r']-base_r
        sensitivity_rows.append({'hypothesis': name, 'variant': label, **m,
            'n_baseline': len(d), 'removed_n': len(d)-len(v),
            'removed_pct': 100*(len(d)-len(v))/len(d),
            'values_modified': label.startswith('winsor'), 'delta_r': delta,
            'sign_flip': np.sign(m['pearson_r']) != np.sign(base_r) if np.isfinite(delta) else False,
            'large_delta_flag': abs(delta) >= CONFIG['sensitivity_delta_r'] if np.isfinite(delta) else False})
    mm = moment_matrix(d, x, y); total = mm.sum(axis=0)
    r_deleted = r_from_moments(total-mm)
    delta_delete = r_deleted-base_r
    xc, yc = mm[:, 1], mm[:, 2]
    b = np.dot(xc, yc)/np.dot(xc, xc)
    e = yc-b*xc; mse = np.dot(e, e)/(len(d)-2)
    leverage = 1/len(d)+xc*xc/np.dot(xc, xc)
    cooks = (e*e/(2*mse))*leverage/np.maximum(1-leverage, 1e-12)**2
    ids = [c for c in ['ccode','iso3','pwt_country','ccode1','ccode2','iso31','iso32','year',
                      'military_growth_pct1','military_growth_pct2','sipri_notes'] if c in d]
    diagnostic = d[ids+list(dict.fromkeys([x,y]))].copy()
    diagnostic.insert(0, 'hypothesis', name)
    diagnostic['x_variable'], diagnostic['y_variable'] = x, y
    diagnostic['r_without_row'] = r_deleted
    diagnostic['delta_r_without_row'] = delta_delete
    diagnostic['cooks_d'] = cooks
    diagnostic['cooks_flag_4_over_n'] = cooks > 4/len(d)
    for method, f in flags.items():
        diagnostic['flag_'+method] = f
    diagnostic['abs_delta'] = np.abs(delta_delete)
    # Exportiere die Vereinigung der wichtigsten Lösch- und Cook-D-Fälle.
    idx = diagnostic.nlargest(CONFIG['top_observations'], 'abs_delta').index.union(
          diagnostic.nlargest(CONFIG['top_observations'], 'cooks_d').index)
    top_rows.append(diagnostic.loc[idx].sort_values('abs_delta', ascending=False))
    outlier_summary.append({'hypothesis': name, 'n': len(d),
        'iqr_flagged': int(flags['iqr'].sum()), 'mad_flagged': int(flags['mad'].sum()),
        'percentile_flagged': int(flags['percentile'].sum()),
        'cooks_flagged': int((cooks > 4/len(d)).sum()),
        'max_single_row_abs_delta_r': float(np.nanmax(np.abs(delta_delete)))})
sensitivity = pd.DataFrame(sensitivity_rows)
report_tables.update(Outlier_sensitivity=sensitivity, Outlier_counts=pd.DataFrame(outlier_summary),
                     Outlier_cutoffs=pd.DataFrame(cutoff_rows),
                     Top_observations=pd.concat(top_rows, ignore_index=True))


In [ ]:
def leave_groups(d, x, y, mode):
    mm = moment_matrix(d, x, y); total = mm.sum(axis=0)
    if mode == 'country' and 'ccode' not in d:
        labels = np.concatenate([d.ccode1.to_numpy(), d.ccode2.to_numpy()])
        moments = np.concatenate([mm, mm])
    else:
        labels = d['ccode' if mode == 'country' else 'year'].to_numpy(); moments = mm
    unique, codes = np.unique(labels, return_inverse=True)
    sums = np.column_stack([np.bincount(codes, weights=moments[:, i], minlength=len(unique)) for i in range(6)])
    rem = total-sums
    out = pd.DataFrame({'excluded_group': unique, 'removed_n': sums[:,0].astype(int),
                        'n_remaining': rem[:,0].astype(int), 'r_without_group': r_from_moments(rem)})
    out['delta_r'] = out.r_without_group-float(r_from_moments(total))
    out['abs_delta_r'] = out.delta_r.abs()
    if mode == 'country':
        out['country_iso3_default'] = out.excluded_group.map(COW_TO_ISO)
    return out.sort_values('abs_delta_r', ascending=False)

leave_country, leave_year = [], []
for name, (d, x, y, _) in datasets.items():
    for mode, rows in [('country', leave_country), ('year', leave_year)]:
        a = leave_groups(d, x, y, mode); a.insert(0, 'hypothesis', name); rows.append(a)
report_tables['Leave_country'] = pd.concat(leave_country, ignore_index=True)
report_tables['Leave_year'] = pd.concat(leave_year, ignore_index=True)

# Country-block bootstrap: ganze Länderreihen, nur H1/H2.
# Kein zeilenweiser Bootstrap für die abhängigen Länderpaare oder überlappende H8-Fenster.
bootstrap_rows = []
rng = np.random.default_rng(CONFIG['random_seed'])
for name in ['H1','H2']:
    d,x,y,_ = datasets[name]; mm = moment_matrix(d,x,y)
    codes, levels = pd.factorize(d.ccode)
    sums = np.column_stack([np.bincount(codes, weights=mm[:,i], minlength=len(levels)) for i in range(6)])
    draws = rng.integers(0, len(levels), size=(CONFIG['bootstrap_repeats'],len(levels)))
    boot = r_from_moments(sums[draws].sum(axis=1))
    finite = boot[np.isfinite(boot)]
    lo,hi = np.quantile(finite,[0.025,0.975])
    bootstrap_rows.append({'hypothesis':name,'resampling_unit':'whole country time series',
        'countries':len(levels),'repeats':len(boot),'valid_repeats':len(finite),
        'pearson_ci_low':lo,'pearson_ci_high':hi,
        'limitation':'Preserves within-country serial dependence; common cross-country shocks remain unaddressed'})
report_tables['Country_bootstrap'] = pd.DataFrame(bootstrap_rows)


### Zeit, Länder, Veränderungen und verzögerte Zusammenhänge

Untergruppen sind ergänzende Analysen. Neue p-Werte sind nicht unabhängige
Bestätigungen. Mittelwerte zwischen Ländern und Abweichungen innerhalb von
Ländern beantworten unterschiedliche Fragen. Erste Differenzen verwenden
ausschließlich exakt aufeinanderfolgende Jahre derselben historischen Einheit.


In [ ]:
time_rows, country_rows, decomposition_rows, lag_rows = [], [], [], []
for name, (d,x,y,_) in datasets.items():
    for grouping in ['year','decade']:
        grouped = d.assign(decade=(d.year//10)*10).groupby(grouping, sort=True)
        for group, v in grouped:
            if len(v) >= CONFIG['min_group_n']:
                time_rows.append({'hypothesis':name,'grouping':grouping,'group':int(group),**corr_metrics(v,x,y)})
    entity = 'ccode' if 'ccode' in d else 'pair_id'
    means = d.groupby(entity)[[x,y]].mean().reset_index()
    decomposition_rows.append({'hypothesis':name,'variant':'between_entity_means',
        'interpretation':'Each entity has equal weight; no causal interpretation',**corr_metrics(means,x,y,False)})
    within = d[[x,y]]-d.groupby(entity)[[x,y]].transform('mean')
    decomposition_rows.append({'hypothesis':name,'variant':'within_entity_demeaned',
        'interpretation':'Within-entity deviations; common time effects not removed',**corr_metrics(within,x,y,False)})
    if entity == 'ccode':
        for code,v in d.groupby('ccode'):
            if len(v) >= CONFIG['min_country_years']:
                country_rows.append({'hypothesis':name,'ccode':code,'iso3':v.iso3.iloc[0],**corr_metrics(v,x,y)})
        prev = d[['ccode','iso3','year',x,y]].copy()
        prev['year'] += 1
        prev = prev.rename(columns={x:x+'_previous',y:y+'_previous'})
        dif = d.merge(prev,on=['ccode','iso3','year'],how='inner',validate='one_to_one')
        dif['dx'] = dif[x]-dif[x+'_previous']; dif['dy'] = dif[y]-dif[y+'_previous']
        decomposition_rows.append({'hypothesis':name,'variant':'exact_annual_first_differences',
            'interpretation':'Consecutive observed years only; original eligible sample',**corr_metrics(dif,'dx','dy',False)})

for name in ['H1','H2']:
    d,x,y,_ = datasets[name]
    target = country_year.loc[country_year.cross_source_eligible & country_year.analysis_period,
                              ['ccode','iso3','year',y]].copy()
    for lag in [0,1,5]:
        future = target.rename(columns={y:'future_y'}).copy(); future['year'] -= lag
        z = complete(d.merge(future,on=['ccode','iso3','year'],validate='one_to_one'),[x,'future_y'])
        lag_rows.append({'hypothesis':name,'lag_years':lag,'sample':'available_for_this_lag',**corr_metrics(z,x,'future_y')})
    # Vergleichbare Stichprobe für alle drei Horizonte.
    matched = d.copy()
    for lag in [0,1,5]:
        future = target.rename(columns={y:'future_'+str(lag)}).copy(); future['year'] -= lag
        matched = matched.merge(future,on=['ccode','iso3','year'],how='inner',validate='one_to_one')
    matched = complete(matched,[x,'future_0','future_1','future_5'])
    for lag in [0,1,5]:
        lag_rows.append({'hypothesis':name,'lag_years':lag,'sample':'same_rows_all_lags',**corr_metrics(matched,x,'future_'+str(lag))})
report_tables.update(Time_groups=pd.DataFrame(time_rows), Country_correlations=pd.DataFrame(country_rows),
                     Within_between=pd.DataFrame(decomposition_rows), Lag_analysis=pd.DataFrame(lag_rows))


### Geografie, bedingte Verteilungen und neue Forschungsfragen

CEPII-Grenzen und Sprachmerkmale sind statische Merkmale und keine vollständig
historisierten Jahresdaten. Die Vergleiche sind entsprechend explorativ.
Echte Null-Handelswerte bleiben erhalten. Relative Asymmetrie ist bei zwei
Nullanteilen nicht definiert und wird nur für positive Gesamthandelsanteile
berechnet. BIP-Größen verwenden PWT cgdpe, nicht nominale Handelswerte.


In [ ]:
trade = datasets['H4'][0].copy()
geo_pair = distances[['iso3_origin','iso3_destination','contig','comlang_off','comlang_ethno']].rename(
    columns={'iso3_origin':'iso31','iso3_destination':'iso32'})
trade = trade.merge(geo_pair,on=['iso31','iso32'],how='left',validate='many_to_one')
for side in [1,2]:
    pp = pwt[['iso3','year','pop','cgdpe']].rename(columns={'iso3':f'iso3{side}',
        'pop':f'pwt_pop{side}','cgdpe':f'pwt_total_gdp{side}'})
    trade = trade.merge(pp,on=[f'iso3{side}','year'],how='left',validate='many_to_one')
trade['log_distance'] = np.log(trade.distance_km.where(trade.distance_km.gt(0)))
trade['share1'] = trade.flow1/trade.imports1; trade['share2'] = trade.flow2/trade.imports2
trade['asymmetry_abs'] = (trade.share1-trade.share2).abs()
trade['asymmetry_relative'] = trade.asymmetry_abs/(trade.share1+trade.share2).where((trade.share1+trade.share2).gt(0))
trade['total_gdp_gap_log'] = (np.log(trade.pwt_total_gdp1.where(trade.pwt_total_gdp1.gt(0)))-
                              np.log(trade.pwt_total_gdp2.where(trade.pwt_total_gdp2.gt(0)))).abs()
trade['mean_log_total_gdp'] = (np.log(trade.pwt_total_gdp1.where(trade.pwt_total_gdp1.gt(0)))+
                               np.log(trade.pwt_total_gdp2.where(trade.pwt_total_gdp2.gt(0))))/2
trade['trade_share_pp'] = 100*trade.mutual_import_share
trade['border_group'] = trade.contig.map({0:'No shared border',1:'Shared border'}).fillna('Unknown')
trade['language_group'] = trade.comlang_off.map({0:'No shared official language',1:'Shared official language'}).fillna('Unknown')
trade['distance_group'] = pd.cut(trade.distance_km,[0,1000,3000,6000,np.inf],
    labels=['0-1000 km','1000-3000 km','3000-6000 km','6000+ km'],include_lowest=True)
trade['wealth_group'] = pd.cut(trade.wealth_gap_log,[0,0.5,1,2,np.inf],
    labels=['0-0.5','0.5-1','1-2','2+'],include_lowest=True)
group_rows=[]
for col in ['border_group','language_group','distance_group','wealth_group']:
    for group,v in trade.groupby(col,observed=True):
        group_rows.append({'grouping':col,'group':str(group),
            'share_mean':v.mutual_import_share.mean(),'share_median':v.mutual_import_share.median(),
            'share_p90':v.mutual_import_share.quantile(0.9),'zero_share_pct':100*v.mutual_import_share.eq(0).mean(),
            'x':'wealth_gap_log','y':'mutual_import_share',**corr_metrics(v,'wealth_gap_log','mutual_import_share')})
report_tables['Trade_groups']=pd.DataFrame(group_rows)
cond=[]
for (border,wealth),v in trade.groupby(['border_group','wealth_group'],observed=True):
    cond.append({'border':border,'wealth_group':str(wealth),'n':len(v),
                 'mean_share':v.mutual_import_share.mean(),'median_share':v.mutual_import_share.median(),
                 'p90_share':v.mutual_import_share.quantile(.9),'positive_share_pct':100*v.mutual_import_share.gt(0).mean()})
report_tables['Conditional_trade']=pd.DataFrame(cond)

urban = country_year.merge(nmc[['ccode','year','upop']],on=['ccode','year'],how='left',validate='one_to_one')
urban = urban.loc[urban.analysis_period & urban.cross_source_eligible].copy()
urban['urban_share'] = urban.upop/urban.tpop.where(urban.tpop.gt(0))
urban.loc[~urban.urban_share.between(0,1),'urban_share']=np.nan
new_specs=[('E1_population_import_concentration',datasets['H2'][0],'tpop','import_hhi_observed','negative'),
           ('E2_urbanisation_military',urban,'urban_share','military_per_1000','negative'),
           ('E3_economic_size_absolute_asymmetry',trade,'total_gdp_gap_log','asymmetry_abs','positive'),
           ('E4_economic_size_relative_asymmetry',trade,'total_gdp_gap_log','asymmetry_relative','positive'),
           ('E5_distance_trade',trade,'log_distance','mutual_import_share','negative')]
report_tables['New_questions']=pd.DataFrame([{'analysis':name,'role':'post-hoc exploratory',
    'x':x,'y':y,'expected_direction':expected,**corr_metrics(d,x,y)} for name,d,x,y,expected in new_specs])

coverage_rows=[]
for low,high in [(.95,1.05),(.90,1.10),(.80,1.20)]:
    # Vergleiche beginnen vor dem Importabdeckungsfilter.
    c2=country_year.loc[country_year.analysis_period & country_year.cross_source_eligible & ~country_year.trade_scope_review]
    c2=c2.loc[c2.import_coverage_ratio.between(low,high)]
    for name,d,x,y in [('H2',c2,'import_hhi_observed','military_per_1000'),
        ('H3',h3_all_coverage.loc[coverage_mask(h3_all_coverage,low,high)],'mutual_import_share','military_growth_gap_pp'),
        ('H4',h4_all_coverage.loc[coverage_mask(h4_all_coverage,low,high)],'wealth_gap_log','mutual_import_share')]:
        coverage_rows.append({'hypothesis':name,'coverage_min':low,'coverage_max':high,
            'role':'baseline' if low==.90 else 'sensitivity_only',**corr_metrics(d,x,y)})
report_tables['Coverage_sensitivity']=pd.DataFrame(coverage_rows)
report_tables['H8_coverage']=h8_coverage_sensitivity.copy()


### Explorative Regressionen mit Länder-/Paar- und Jahreseffekten

Implementierung mit NumPy, keine zusätzliche statsmodels-Installation nötig.
Gruppeneffekte werden durch alternierendes Zentrieren absorbiert. Standardfehler
verwenden eine zweifach geclusterte Sandwich-Kovarianz mit kleinen
Stichprobenkorrekturen. t-Referenz: min(Clusteranzahlen)-1 Freiheitsgrade.
Das ist keine kausale Identifikation. Paar-/Jahrescluster erfassen NICHT sämtliche
Abhängigkeiten zwischen verschiedenen Paaren mit demselben Land. Entsprechende
p-Werte bleiben explorativ. Zeitkonstante Geografiemerkmale werden nur im Modell
mit Jahreseffekten geschätzt, nicht bei absorbierten Paareffekten.


In [ ]:
def residualize(matrix, grouping_arrays, tolerance=1e-10, max_iter=1000):
    z=np.asarray(matrix,float).copy()
    # Gemeinsame Standardisierung pro Spalte verbessert die Konvergenzprüfung.
    scale=np.std(z,axis=0); scale[scale==0]=1
    z/=scale
    codes=[pd.factorize(v)[0] for v in grouping_arrays]
    for iteration in range(max_iter):
        old=z.copy()
        for c in codes:
            count=np.bincount(c)
            for j in range(z.shape[1]):
                means=np.bincount(c,weights=z[:,j])/count
                z[:,j]-=means[c]
        if np.max(np.abs(z-old))<tolerance:
            return z*scale,iteration+1
    raise RuntimeError('Fixed-effect demeaning did not converge; no model silently exported.')

def fit_absorbed(name,d,y,xcols,entity_col,entity_fe=False,limitation=''):
    needed=list(dict.fromkeys([y]+xcols+[entity_col,'year']))
    a=complete(d, [y]+xcols).dropna(subset=[entity_col,'year']).reset_index(drop=True)
    entities=a[entity_col].nunique(); years=a.year.nunique()
    groups=[a[entity_col].to_numpy(),a.year.to_numpy()] if entity_fe else [a.year.to_numpy()]
    z,iterations=residualize(a[[y]+xcols].to_numpy(),groups)
    yy=z[:,0]; xx=z[:,1:]; n,p=xx.shape
    if n<=p+entities+years or entities<3 or years<3:
        raise ValueError(name+': insufficient rows or clusters')
    if np.linalg.matrix_rank(xx)<p:
        raise ValueError(name+': collinear regressors after fixed effects')
    beta=np.linalg.lstsq(xx,yy,rcond=None)[0]; residual=yy-xx@beta
    bread=np.linalg.inv(xx.T@xx); scores=xx*residual[:,None]
    # Für diese Daten überprüfte zusammenhängende Entity-Year-Graphen:
    # FE-Rang bei zwei absorbierten Effekten = G_entity+G_year-1.
    if entity_fe:
        ec=pd.factorize(a[entity_col])[0]; yc=pd.factorize(a.year)[0]
        parent=np.arange(entities+years)
        def root(i):
            while parent[i]!=i:
                parent[i]=parent[parent[i]]; i=parent[i]
            return i
        for e,t in zip(ec,yc):
            re,rt=root(int(e)),root(entities+int(t))
            if re!=rt: parent[rt]=re
        components=len({root(i) for i in range(len(parent))})
        fe_rank=entities+years-components
    else:
        fe_rank=years; components=1
    k=p+fe_rank
    def cluster_cov(labels):
        code,level=pd.factorize(labels)
        g=len(level)
        sums=np.column_stack([np.bincount(code,weights=scores[:,j],minlength=g) for j in range(p)])
        return (bread@(sums.T@sums)@bread)*(g/(g-1))*((n-1)/(n-k))
    joint=pd.MultiIndex.from_frame(a[[entity_col,'year']])
    covariance=cluster_cov(a[entity_col])+cluster_cov(a.year)-cluster_cov(joint)
    variances=np.diag(covariance)
    se=np.where(variances>=0,np.sqrt(np.maximum(variances,0)),np.nan)
    tvalues=np.divide(beta,se,out=np.full(p,np.nan),where=se>0)
    dof=min(entities,years)-1
    pvalues=2*stats.t.sf(np.abs(tvalues),dof)
    critical=stats.t.ppf(.975,dof)
    rows=[]
    for i,col in enumerate(xcols):
        rows.append({'model':name,'term':col,'coefficient':beta[i],'se_cluster_2way':se[i],
            'ci_low':beta[i]-critical*se[i],'ci_high':beta[i]+critical*se[i],
            'p_cluster_exploratory':pvalues[i],'n':n,'clusters_entity':entities,'clusters_year':years,
            'entity_fixed_effects':entity_fe,'year_fixed_effects':True,'limitation':limitation})
    detail={'model':name,'outcome':y,'n_input':len(d),'n_used':n,'removed_incomplete':len(d)-n,
            'within_r_squared':1-np.dot(residual,residual)/np.dot(yy,yy),
            'absorbed_effect_rank':fe_rank,'connected_components':components,'demeaning_iterations':iterations,
            't_reference_df':dof,'negative_covariance_diagonal_count':int((variances<0).sum()),
            'se_method':'entity + year cluster sandwich, inclusion-exclusion, finite-sample correction',
            'interpretation':'Associations conditional on included covariates/effects; no causality'}
    return rows,detail

model_rows,model_details=[],[]
d1=datasets['H1'][0].copy(); d1['log_productivity']=np.log(d1.productivity)
d2=datasets['H2'][0].copy(); d2['log_productivity']=np.log(d2.productivity.where(d2.productivity.gt(0)))
d2['log_population']=np.log(d2.tpop.where(d2.tpop.gt(0)))
tr=trade.copy(); tr['wealth_x_logdistance']=tr.wealth_gap_log*tr.log_distance
tr['language_x_logdistance']=tr.comlang_off*tr.log_distance
model_specs=[
    ('R1_H1_country_year_FE',d1,'military_per_1000',['log_productivity'],'ccode',True,
     'Country/year clustering; cross-country dependence beyond common year clusters may remain'),
    ('R2_H2_country_year_FE',d2,'military_per_1000',['import_hhi_observed','log_productivity','log_population'],'ccode',True,
     'Additional complete-case sample; coefficient is conditional, not causal'),
    ('R3_trade_geography_year_FE',tr,'trade_share_pp',
     ['wealth_gap_log','log_distance','contig','comlang_off','mean_log_total_gdp','total_gdp_gap_log'],
     'pair_id',False,'Static CEPII attributes; pair/year clusters do not cover all shared-country dyadic dependence'),
    ('R4_trade_interactions_year_FE',tr,'trade_share_pp',
     ['wealth_gap_log','log_distance','contig','comlang_off','mean_log_total_gdp','total_gdp_gap_log',
      'wealth_x_logdistance','language_x_logdistance'],
     'pair_id',False,'Interaction models are post-hoc; pair/year clusters do not cover all dyadic dependence'),
    ('R5_trade_pair_year_FE',tr,'trade_share_pp',
     ['wealth_gap_log','mean_log_total_gdp','total_gdp_gap_log'],'pair_id',True,
     'Within-pair/year associations; shared-country dyadic dependence can remain'),
]
for args in model_specs:
    rows,detail=fit_absorbed(*args); model_rows.extend(rows); model_details.append(detail)
# Dieselben zwei Modelle zusätzlich mit festgelegten Ausreißersensitivitäten.
for variant in ['trim_p01_p99','winsor_p01_p99']:
    d=datasets['H1'][0].copy()
    for col in ['productivity','military_per_1000']:
        bound=next(v for v in cutoff_rows if v['hypothesis']=='H1' and v['method']=='percentile' and v['variable']==col)
        if variant.startswith('trim'):
            d=d.loc[d[col].between(bound['low'],bound['high'])].copy()
        else:
            d[col]=d[col].clip(bound['low'],bound['high'])
    d['log_productivity']=np.log(d.productivity)
    rows,detail=fit_absorbed('R1_'+variant,d,'military_per_1000',['log_productivity'],'ccode',True,
        'Sensitivity only; full original model remains reference')
    model_rows.extend(rows); model_details.append(detail)
    d=trade.copy()
    for col in ['wealth_gap_log','mutual_import_share']:
        bound=next(v for v in cutoff_rows if v['hypothesis']=='H4' and v['method']=='percentile' and v['variable']==col)
        if variant.startswith('trim'):
            d=d.loc[d[col].between(bound['low'],bound['high'])].copy()
        else:
            d[col]=d[col].clip(bound['low'],bound['high'])
    d['trade_share_pp']=100*d.mutual_import_share
    rows,detail=fit_absorbed('R3_'+variant,d,'trade_share_pp',
        ['wealth_gap_log','log_distance','contig','comlang_off','mean_log_total_gdp','total_gdp_gap_log'],
        'pair_id',False,'Sensitivity only; pair/year clusters do not capture all shared-country dependence')
    model_rows.extend(rows); model_details.append(detail)
report_tables['Regression']=pd.DataFrame(model_rows)
report_tables['Model_details']=pd.DataFrame(model_details)


### Grafiken und nachvollziehbare Zusammenfassung

Die vollständige Stichprobe bleibt die Referenz. Eine vergrößerte Ansicht zeigt
nur einen Ausschnitt der Achsen; sie ersetzt keine Hauptanalyse. Farben in
Hexbin-Plots zeigen die logarithmierte Anzahl von Beobachtungen pro Sechseck.


In [ ]:
def add_figure(sheet,title,fig):
    buf=BytesIO(); fig.savefig(buf,format='png',dpi=140,bbox_inches='tight',facecolor='white')
    chart_items.append({'sheet':sheet,'title':title,'png':buf.getvalue()})
    plt.close(fig)

def density(ax,d,x,y,title):
    v=complete(d,[x,y])
    if len(v)<3 or v[x].nunique()<2 or v[y].nunique()<2:
        ax.text(.5,.5,'Insufficient variation',ha='center',transform=ax.transAxes); return
    im=ax.hexbin(v[x],v[y],gridsize=40,mincnt=1,bins='log',cmap='viridis',rasterized=True)
    ax.set(xlabel=LABELS.get(x,x),ylabel=LABELS.get(y,y),title=title)
    ax.figure.colorbar(im,ax=ax,label='Observation count',shrink=.8)

for name,(d,x,y,_) in datasets.items():
    r=baseline.set_index('hypothesis').loc[name,'pearson_r']
    fig,axs=plt.subplots(2,2,figsize=(13,9),layout='constrained')
    density(axs[0,0],d,x,y,f'{name}: full sample, n={len(d):,}, r={r:.3f}')
    zoom=variant_frames[name]['trim_p01_p99']
    density(axs[0,1],zoom,x,y,'Central percentile view only; full sample retained')
    v=sensitivity.loc[sensitivity.hypothesis.eq(name)]
    pos=np.arange(len(v))
    axs[1,0].barh(pos-.17,v.pearson_r,height=.32,label='Pearson')
    axs[1,0].barh(pos+.17,v.spearman_rho,height=.32,label='Spearman',color='#d18b35')
    axs[1,0].set_yticks(pos,v.variant.str.replace('_',' '),fontsize=8)
    axs[1,0].axvline(0,color='grey',lw=.7); axs[1,0].set(xlabel='Correlation',title='Outlier sensitivity')
    axs[1,0].legend(fontsize=8)
    t=report_tables['Time_groups']; t=t.loc[t.hypothesis.eq(name)&t.grouping.eq('year')]
    axs[1,1].plot(t.group,t.pearson_r,label='Pearson')
    axs[1,1].plot(t.group,t.spearman_rho,label='Spearman',color='#d18b35')
    axs[1,1].axhline(0,color='grey',lw=.7); axs[1,1].set(xlabel='Year',ylabel='Correlation',title='Annual correlations')
    axs[1,1].legend(fontsize=8)
    add_figure('Charts_'+name,name+' overview',fig)

fig,axs=plt.subplots(1,2,figsize=(13,4.5),layout='constrained')
d,x,y,_=datasets['H1']; m=d.groupby('ccode')[[x,y]].mean().reset_index()
density(axs[0],m,x,y,'H1: between country means')
w=d[[x,y]]-d.groupby('ccode')[[x,y]].transform('mean')
density(axs[1],w,x,y,'H1: within country deviations (axes show deviations)')
add_figure('Charts_differences','Between and within country comparison',fig)

fig,axs=plt.subplots(1,2,figsize=(13,4.5),layout='constrained')
g=report_tables['Trade_groups']; v=g.loc[g.grouping.eq('wealth_group')]
axs[0].plot(v.group,100*v.share_mean,marker='o',label='Mean')
axs[0].plot(v.group,100*v.share_median,marker='o',label='Median')
axs[0].plot(v.group,100*v.share_p90,marker='o',label='90th percentile')
axs[0].set(xlabel='Absolute log GDPpc gap',ylabel='Mutual import share (%)',title='Trade distribution by wealth gap'); axs[0].legend()
density(axs[1],trade,'total_gdp_gap_log','asymmetry_abs','Economic size and absolute import dependence gap')
add_figure('Charts_trade','Trade distributions and asymmetry',fig)

fig,axs=plt.subplots(1,2,figsize=(13,4.5),layout='constrained')
geography_bin_rows=[]
finite_distance=trade.log_distance.dropna()
common_distance_bins=np.linspace(finite_distance.min(),finite_distance.max(),11)
for i,(col,title) in enumerate([('border_group','Shared border'),('language_group','Shared official language')]):
    for group,v in trade.groupby(col):
        if group=='Unknown': continue
        bins=pd.cut(v.log_distance,common_distance_bins,include_lowest=True)
        a=v.groupby(bins,observed=True).agg(distance=('distance_km','median'),mean_share=('trade_share_pp','mean'),n=('year','size'))
        for interval,row in a.iterrows():
            geography_bin_rows.append({'grouping':col,'group':group,'log_distance_bin_low':interval.left,
                'log_distance_bin_high':interval.right,'median_distance_km':row['distance'],
                'mean_mutual_share_pct':row['mean_share'],'n':int(row['n']),
                'shown_in_chart':row['n']>=CONFIG['min_group_n']})
        a=a.loc[a.n.ge(CONFIG['min_group_n'])]
        axs[i].plot(a.distance,a.mean_share,marker='o',label=group)
    axs[i].set(xscale='log',xlabel='Median distance per bin (km, log scale)',ylabel='Mean mutual import share (%)',title=title)
    axs[i].legend(fontsize=8)
add_figure('Charts_trade','Geographic comparison using common log-distance bins',fig)
report_tables['Geographic_bins']=pd.DataFrame(geography_bin_rows)

# Länderjahre mit größten absoluten Änderungen in den H3-Paaren.
extremes=pd.concat([datasets['H3'][0][[f'ccode{s}',f'iso3{s}','year',f'military_growth_pct{s}']].rename(
    columns={f'ccode{s}':'ccode',f'iso3{s}':'iso3',f'military_growth_pct{s}':'military_growth_pct'}) for s in [1,2]])
extremes=extremes.drop_duplicates(['ccode','year']); extremes['abs_growth']=extremes.military_growth_pct.abs()
extremes=extremes.sort_values('abs_growth',ascending=False).head(50)
extremes=extremes.merge(country_year[['ccode','year','military_spending_real_musd','sipri_notes']],
                       on=['ccode','year'],how='left',validate='one_to_one')
previous=country_year[['ccode','iso3','year','military_spending_real_musd']].copy(); previous.year+=1
previous=previous.rename(columns={'military_spending_real_musd':'previous_year_spending_musd'})
extremes=extremes.merge(previous,on=['ccode','iso3','year'],how='left',validate='one_to_one')
report_tables['H3_extreme_years']=extremes
top_codes=extremes.ccode.drop_duplicates().head(3).tolist()
fig,axs=plt.subplots(len(top_codes),2,figsize=(13,3*len(top_codes)),squeeze=False,layout='constrained')
for i,code in enumerate(top_codes):
    v=country_year.loc[country_year.ccode.eq(code)&country_year.analysis_period&country_year.cross_source_eligible].sort_values('year')
    axs[i,0].plot(v.year,v.military_spending_real_musd); axs[i,0].set(title=str(COW_TO_ISO.get(code,code)),ylabel='Real spending (million 2024 USD)',xlabel='Year')
    axs[i,1].plot(v.year,v.military_growth_pct); axs[i,1].set(ylabel='Annual growth (%)',xlabel='Year')
add_figure('Charts_influence','H3: spending series behind extreme growth rates',fig)

fig,axs=plt.subplots(1,2,figsize=(13,5),layout='constrained')
for i,name in enumerate(['H1','H3']):
    v=report_tables['Leave_country'].loc[lambda z:z.hypothesis.eq(name)].head(10).iloc[::-1]
    axs[i].barh(v.country_iso3_default.fillna(v.excluded_group.astype(str)),v.delta_r)
    axs[i].axvline(0,color='grey',lw=.7); axs[i].set(title=name+': exclude each country and all its pairs',xlabel='Change in Pearson r')
add_figure('Charts_influence','Country influence',fig)

fig,axs=plt.subplots(1,2,figsize=(13,5),layout='constrained')
reg=report_tables['Regression']
for ax,prefix,term,title in [(axs[0],'R1_','log_productivity','H1: productivity coefficient with country/year FE'),
                            (axs[1],'R3_','wealth_gap_log','Trade: wealth-gap coefficient with year FE')]:
    v=reg.loc[reg.model.str.startswith(prefix)&reg.term.eq(term)].copy()
    pos=np.arange(len(v))
    ax.errorbar(v.coefficient,pos,xerr=np.vstack([v.coefficient-v.ci_low,v.ci_high-v.coefficient]),fmt='o',capsize=4)
    ax.set_yticks(pos,v.model.str.replace('_',' '),fontsize=8)
    ax.axvline(0,color='grey',lw=.7)
    ax.set(title=title,xlabel='Coefficient and exploratory 95% clustered interval')
add_figure('Charts_models','Regression outlier sensitivity; see inference limitations in Methods',fig)


In [ ]:
summary_rows=[]
for row in baseline.to_dict('records'):
    name=row['hypothesis']; v=sensitivity.loc[sensitivity.hypothesis.eq(name)&sensitivity.variant.ne('full')]
    lc=report_tables['Leave_country'].loc[lambda z:z.hypothesis.eq(name)].iloc[0]
    ly=report_tables['Leave_year'].loc[lambda z:z.hypothesis.eq(name)].iloc[0]
    largest=v.loc[v.delta_r.abs().idxmax()]
    summary_rows.append({'hypothesis':name,'n':row['n'],'pearson_r':row['pearson_r'],
        'spearman_rho':row['spearman_rho'],'p_standard':row['p_standard'],
        'minimum_5000_met':row['minimum_5000_met'],
        'direction_matches_expectation':row['observed_direction_matches'],
        'largest_sensitivity_variant':largest['variant'],'largest_sensitivity_delta_r':largest['delta_r'],
        'most_influential_country':lc['country_iso3_default'],'country_exclusion_delta_r':lc['delta_r'],
        'most_influential_year':ly['excluded_group'],'year_exclusion_delta_r':ly['delta_r'],
        'interpretation':'Exploratory only; original hypothesis retained; outlier flags are not data errors'})
report_tables={'Summary':pd.DataFrame(summary_rows),**report_tables}
report_tables['Audit']=audit.copy()
report_tables['Variables']=data_dictionary.copy()
report_tables['Additional_variables']=pd.DataFrame([
    ('asymmetry_abs','abs(flow1/imports1-flow2/imports2)','0-1'),
    ('asymmetry_relative','abs(share1-share2)/(share1+share2); undefined if both zero','0-1'),
    ('total_gdp_gap_log','abs(log(PWT cgdpe1)-log(PWT cgdpe2))','log units'),
    ('urban_share','NMC urban population / total population','0-1'),
    ('trade_share_pp','100*mutual_import_share','percent'),
    ('delta_r','variant r minus complete baseline r','correlation difference'),
],columns=['variable','definition','unit'])
report_tables['Sources']=source_catalog.copy()
foot=metadata['sipri_footnotes'].dropna(how='all').copy()
report_tables['SIPRI_source_notes']=pd.DataFrame([
    {'original_excel_row':int(i)+1,'reference':row.iloc[0],
     'note_text':' '.join(str(v) for v in row.iloc[1:] if pd.notna(v))}
    for i,row in foot.iterrows()])
report_tables['Parameters']=pd.DataFrame([{'parameter':k,'value':str(v)} for k,v in CONFIG.items()]+[
    {'parameter':'start_year','value':str(START_YEAR)}, {'parameter':'end_year','value':str(END_YEAR)},
    {'parameter':'analysis_code_sha256','value':ANALYSIS_CODE_SHA256},
    {'parameter':'run_started_berlin','value':run_started.isoformat()},
    {'parameter':'python','value':platform.python_version()}, {'parameter':'pandas','value':pd.__version__},
    {'parameter':'numpy','value':np.__version__}, {'parameter':'scipy','value':scipy.__version__},
    {'parameter':'matplotlib','value':matplotlib.__version__}, {'parameter':'openpyxl','value':openpyxl.__version__},
])
method_text=[
    ('Scope','Original H1-H4; H8 and all extensions are exploratory. No hypothesis is changed to obtain significance.'),
    ('Baseline coverage','H3/H4 require import coverage [0.90,1.10] in BOTH countries. This corrects the earlier preparation sample.'),
    ('Rows versus independence','n counts complete rows, not independent countries or wars. Minimum 5000 is a course criterion, not an independence test.'),
    ('Pearson p','p_standard is the conventional two-sided SciPy Pearson p-value. Serial and shared-country dependence are not accounted for.'),
    ('Many analyses','Multiple exploratory comparisons are shown. No automatic significance/confirmation labels; no uncorrected search for small p-values.'),
    ('Zero p-values','A stored zero p-value can mean numerical underflow and must not be interpreted as exactly zero probability.'),
    ('Outlier rules','IQR=1.5*IQR; robust MAD-z=3.5; trim/winsor=1st/99th percentile of each variable in baseline. Zero IQR/MAD disables that variable rule.'),
    ('Trimming','Excludes rows outside either X or Y cutoff. All variants are sensitivities, never replacements for the original full-sample estimate.'),
    ('Winsorising','Clips X and Y to cutoffs while retaining rows. Changes values, not the number of observations.'),
    ('Cook distance','Simple y~constant+x OLS only, flag 4/n. Descriptive screen, not evidence of a measurement error.'),
    ('Row influence','Exact Pearson correlation after removing one observation; exported top cases are a selection, not all flagged observations.'),
    ('Country influence','Removes all rows involving that country, including BOTH pair roles. More important than single-row deletion for repeated shocks.'),
    ('Bootstrap','H1/H2 only, resample entire country time series. Common cross-country shocks may remain. Not applied to overlapping dyads/H8 windows.'),
    ('Time groups','Annual and decade correlations use available complete data, minimum 30 pairs. Composition changes can explain differences.'),
    ('Within/between','Demeaned correlations are descriptive; p suppressed. Country means give equal entity weight; within effects retain observation weights.'),
    ('Lags','Exact t+1 and t+5 targets; both time points eligible, within 1950-2014 and same ISO identity. Same-rows comparisons are also included.'),
    ('Geography','CEPII language/border attributes are static, not historically year-specific. Country-scope exclusions from preparation remain in place.'),
    ('Distance chart bins','Ten equal-width bins of log distance using common edges for both comparison groups; plotted only for bins with at least 30 observations.'),
    ('Asymmetry','Absolute import-share difference includes genuine bilateral zeros. Relative measure excludes two-zero cases; inspect sample-size difference.'),
    ('H3 extremes','Growth can be very large due to a small prior-year denominator. Spending levels and original SIPRI notes require review before exclusion.'),
    ('SIPRI quality','Estimate/uncertainty cell colours are not fully converted to flags. Read the original source notes; no automated validity claim.'),
    ('H8','Exploratory, below 5000 baseline rows. Modulo-5 sensitivity reduces overlapping windows but does not remove all dependence.'),
    ('Regression','OLS estimated on absorbed group/year residuals. Two-way cluster covariance uses entity+year minus intersection, finite-sample correction.'),
    ('Regression inference','Dyad/year clustering does not capture every dependence between different dyads sharing a country. All inference is exploratory.'),
    ('Covariance','Negative diagonal estimates from the two-way sandwich are reported as missing SEs, never silently forced positive.'),
    ('Interpretation','Correlations/regressions describe associations. No causal conclusions, automatic outlier deletion, or hypothesis confirmation.'),
    ('Files','One timestamped XLSX with embedded PNGs. All analysis tables are numeric/structured. Sources have SHA256 hashes. The notebook is the executable method.'),
]
report_tables={'Summary':report_tables.pop('Summary'),'Methods':pd.DataFrame(method_text,columns=['topic','explanation']),**report_tables}
display(report_tables['Summary'])
print('Analysen abgeschlossen:',len(report_tables),'Tabellen;',len(chart_items),'Grafiken.')


## Kriegsstatus, Kontinente und individuelle Extremfälle

Dieser Teil erweitert die bisherigen Analysen. Die vorhandene COW-Datei
erfasst **zwischenstaatliche Kriege bis 2007**. Keine erfasste Beteiligung
bedeutet daher weder Bürgerkriegsfreiheit noch allgemein Frieden. Ab 2008
bleibt der Kriegsstatus unbekannt. Ein Länderpaar mit zwei beteiligten Staaten
muss nicht gegeneinander kämpfen.

Alle Grafiken werden in EINEM neuen Excel-Bericht gespeichert, je Hypothese
und darin nach Gruppierung. Direkt darunter stehen Kennzahlen und kurze
datenbasierte Hinweise. Tabellen zu Fällen und Modellen bleiben maschinenlesbar.
Die Vorgängeranalysen und ihre Grafiken werden im selben Bericht mitgeführt.


In [ ]:
import textwrap
from collections import defaultdict
from PIL import Image as PILImage

WAR_CONFIG = {
    'war_coverage_start': 1816, 'war_coverage_end': 2007,
    'minimum_frequency_years': 10,
    'group_min_n': 30, 'tail_rows_per_direction': 3,
    'chart_dpi': 105,
    'include_crossed_war_continent': True,
    'manual_exclude_iso3': [],
}
WAR_CODE_SHA256 = '0120536a53a2581adb0ecd7bf03dd1e4d01217f36bfaf9d4719d87b0b0275fbe'
CONTINENTS = ['Afrika','Asien','Europa','Nordamerika','Südamerika','Ozeanien']
SOUTH_AMERICA_ISO = {'ARG','BOL','BRA','CHL','COL','ECU','GUY','PRY','PER','SUR','URY','VEN','GUF','FLK'}
CONTINENT_MAP = {'Africa':'Afrika','Asia':'Asien','Europe':'Europa','Pacific':'Ozeanien'}

continent_lookup=geography[['iso3','country','continent']].copy()
continent_lookup=continent_lookup.rename(columns={'continent':'cepii_continent'})
continent_lookup['continent']=continent_lookup.cepii_continent.map(CONTINENT_MAP)
american=continent_lookup.cepii_continent.eq('America')
continent_lookup.loc[american,'continent']='Nordamerika'
continent_lookup.loc[american & continent_lookup.iso3.isin(SOUTH_AMERICA_ISO),'continent']='Südamerika'
continent_lookup['continent']=continent_lookup.continent.fillna('Unbekannt')
continent_lookup['classification_note']=np.where(american,
    'CEPII America split using explicit South America ISO list; remainder North America including Central America/Caribbean',
    'CEPII classification; Pacific renamed Oceania; transcontinental states assigned once')
require_unique(continent_lookup,['iso3'],'Continent mapping')

def episodes_from_wars(frame, coverage_end):
    episodes,issues=[],[]
    for source_row,row in frame.iterrows():
        for episode in [1,2]:
            fields=[f'{part}{episode}' for part in ['StartMonth','StartDay','StartYear','EndMonth','EndDay','EndYear']]
            vals=pd.to_numeric(row[fields],errors='coerce')
            if episode==2 and vals.eq(-8).all(): continue
            sy,ey=vals.iloc[2],vals.iloc[5]
            ongoing=ey==-7
            if ongoing: ey=coverage_end
            valid=pd.notna(sy) and pd.notna(ey) and sy>0 and ey>0 and ey>=sy
            record={'source_row':int(source_row)+2,'war_id':int(row.WarNum),'war_name':row.WarName,
                'ccode':int(row.ccode),'state_name_raw':row.StateName,'side_raw':int(row.Side),
                'episode':episode,'start_year':int(sy) if pd.notna(sy) and sy>0 else np.nan,
                'end_year':int(ey) if pd.notna(ey) and ey>0 else np.nan,
                'ongoing_at_source_end':bool(ongoing),'valid_year_bounds':bool(valid)}
            if valid:
                episodes.append(record)
            else:
                # Unbestimmte Episode bleibt sichtbar. Betroffene mögliche Jahre
                # werden später nicht als kriegsfrei codiert.
                issues.append({**record,'reason':'unknown_or_invalid_episode_year_bounds'})
    cols=['source_row','war_id','war_name','ccode','state_name_raw','side_raw','episode',
          'start_year','end_year','ongoing_at_source_end','valid_year_bounds']
    return pd.DataFrame(episodes,columns=cols),pd.DataFrame(issues,columns=cols+['reason'])

war_episodes,war_date_issues=episodes_from_wars(raw['wars'],WAR_CONFIG['war_coverage_end'])
annual_records=[]
for row in war_episodes.itertuples(index=False):
    for year in range(max(int(row.start_year),WAR_CONFIG['war_coverage_start']),
                      min(int(row.end_year),WAR_CONFIG['war_coverage_end'])+1):
        annual_records.append({'ccode':row.ccode,'year':year,'war_id':row.war_id,'war_name':row.war_name})
war_participation_years=pd.DataFrame(annual_records).drop_duplicates(['ccode','year','war_id'])
annual_wars=war_participation_years.groupby(['ccode','year'],as_index=False).agg(
    interstate_wars_n=('war_id','nunique'),
    war_ids=('war_id',lambda s:','.join(str(v) for v in sorted(set(s)))),
    war_names=('war_name',lambda s:'; '.join(sorted(set(s)))))

war_country_year=country_year[['ccode','iso3','year','in_cow_system','cross_source_eligible']].copy()
war_country_year=war_country_year.merge(annual_wars,on=['ccode','year'],how='left',validate='one_to_one')
war_country_year['war_known']=war_country_year.year.between(WAR_CONFIG['war_coverage_start'],WAR_CONFIG['war_coverage_end']) & war_country_year.in_cow_system
for row in war_date_issues.itertuples(index=False):
    low=int(row.start_year) if pd.notna(row.start_year) else WAR_CONFIG['war_coverage_start']
    high=int(row.end_year) if pd.notna(row.end_year) else WAR_CONFIG['war_coverage_end']
    war_country_year.loc[war_country_year.ccode.eq(row.ccode)&war_country_year.year.between(low,high),'war_known']=False
war_country_year['interstate_wars_n']=war_country_year.interstate_wars_n.fillna(0).where(war_country_year.war_known)
war_country_year['at_interstate_war']=war_country_year.interstate_wars_n.gt(0).astype('boolean').where(war_country_year.war_known,pd.NA)
for col in ['war_ids','war_names']:
    war_country_year[col]=war_country_year[col].fillna('').where(war_country_year.war_known,'Unbekannter Status')
war_country_year=war_country_year.merge(continent_lookup[['iso3','continent','country']],on='iso3',how='left',validate='many_to_one')
war_country_year['continent']=war_country_year.continent.fillna('Unbekannt')
war_country_year['country_label']=war_country_year.country.fillna(war_country_year.iso3).fillna(war_country_year.ccode.astype(str))
require_unique(war_country_year,['ccode','year'],'War country-year')

# Kriegshäufigkeit unabhängig von vollständigen X/Y-Paaren der Hypothesen.
freq_base=war_country_year.loc[war_country_year.year.between(START_YEAR,min(END_YEAR,WAR_CONFIG['war_coverage_end'])) &
    war_country_year.cross_source_eligible & war_country_year.war_known].copy()
frequency_profile=freq_base.groupby('iso3',as_index=False).agg(
    country=('country_label','first'),continent=('continent','first'),
    observed_years=('year','nunique'),war_years=('at_interstate_war','sum'),
    first_observed_year=('year','min'),last_observed_year=('year','max'))
distinct=freq_base[['ccode','iso3','year']].merge(war_participation_years,on=['ccode','year'],how='left',validate='one_to_many')
distinct=distinct.groupby('iso3').war_id.nunique().rename('distinct_interstate_wars')
frequency_profile=frequency_profile.merge(distinct,on='iso3',how='left',validate='one_to_one')
frequency_profile['war_year_share']=frequency_profile.war_years.astype(float)/frequency_profile.observed_years
frequency_profile['wars_per_10_observed_years']=10*frequency_profile.distinct_interstate_wars/frequency_profile.observed_years
frequency_profile['frequency_eligible']=frequency_profile.observed_years.ge(WAR_CONFIG['minimum_frequency_years'])
ref=frequency_profile.loc[frequency_profile.frequency_eligible]
frequency_mean_share=ref.war_year_share.mean()
frequency_q90_share=ref.war_year_share.quantile(.9)
frequency_mean_count=ref.distinct_interstate_wars.mean()
for name,mask in {
    'above_mean_share':frequency_profile.war_year_share.gt(frequency_mean_share),
    'above_p90_share':frequency_profile.war_year_share.gt(frequency_q90_share),
    'above_mean_count':frequency_profile.distinct_interstate_wars.gt(frequency_mean_count),
}.items():
    frequency_profile[name]=mask.astype('boolean').where(frequency_profile.frequency_eligible,pd.NA)
frequency_profile=frequency_profile.sort_values(['war_year_share','distinct_interstate_wars'],ascending=False).reset_index(drop=True)
war_country_year=war_country_year.merge(frequency_profile[['iso3','observed_years','war_year_share','distinct_interstate_wars',
    'frequency_eligible','above_mean_share','above_p90_share','above_mean_count']],on='iso3',how='left',validate='many_to_one')

def attach_context(df):
    result=df.copy()
    fields=['ccode','year','at_interstate_war','war_known','war_names','continent','country_label',
        'observed_years','war_year_share','distinct_interstate_wars','frequency_eligible',
        'above_mean_share','above_p90_share','above_mean_count']
    if 'ccode' in result:
        result=result.merge(war_country_year[fields],on=['ccode','year'],how='left',validate='one_to_one')
        result['war_group']=np.where(result.war_known.fillna(False),
            np.where(result.at_interstate_war.fillna(False),'Beteiligung erfasst','Keine Beteiligung erfasst'),'Status unbekannt')
        result['war_any']=result.at_interstate_war.astype('Float64')
    else:
        for side in [1,2]:
            ctx=war_country_year[fields].rename(columns={c:f'{c}_{side}' for c in fields if c!='year'})
            ctx=ctx.rename(columns={f'ccode_{side}':f'ccode{side}'})
            result=result.merge(ctx,on=[f'ccode{side}','year'],how='left',validate='many_to_one')
        known=result.war_known_1.fillna(False)&result.war_known_2.fillna(False)
        count=result.at_interstate_war_1.astype('Float64')+result.at_interstate_war_2.astype('Float64')
        result['war_countries_n']=count.where(known)
        result['war_known']=known
        result['war_group']=count.map({0:'Keines beteiligt',1:'Ein Land beteiligt',2:'Beide beteiligt'}).fillna('Status unbekannt')
        result['war_any']=count.gt(0).astype('Float64').where(known)
        result['war_both']=count.eq(2).astype('Float64').where(known)
        a=result.continent_1.fillna('Unbekannt');b=result.continent_2.fillna('Unbekannt')
        result['continent_pair']=np.where(a.le(b),a+' / '+b,b+' / '+a)
        result['continent_relation']=np.where(result.continent_1.eq('Unbekannt')|result.continent_2.eq('Unbekannt'),'Unbekannt',
            np.where(result.continent_1.eq(result.continent_2),'Gleicher Kontinent','Verschiedene Kontinente'))
    return result

all_specs={}
for key,(df,x,y,expected) in datasets.items():
    all_specs[key]={'df':attach_context(df),'x':x,'y':y,'expected':expected,'military':key in ['H1','H2','H3']}
for old_key,df,x,y,expected in new_specs:
    key=old_key.split('_')[0]
    d=complete(df,[x,y]).reset_index(drop=True)
    if 'ccode' not in d and 'pair_id' not in d:
        d['pair_id']=d.ccode1.astype(str)+'_'+d.ccode2.astype(str)
    all_specs[key]={'df':attach_context(d),'x':x,'y':y,'expected':expected,'military':key=='E2'}

# Zeitraum t-1/t für H3, da die Ausgabenänderung beide Jahre verwendet.
h3_ctx=all_specs['H3']['df']
for side in [1,2]:
    previous=war_country_year[['ccode','year','at_interstate_war','war_known']].copy(); previous.year+=1
    previous=previous.rename(columns={'ccode':f'ccode{side}','at_interstate_war':f'war_previous_{side}',
                                      'war_known':f'previous_known_{side}'})
    h3_ctx=h3_ctx.merge(previous,on=[f'ccode{side}','year'],how='left',validate='many_to_one')
window_known=h3_ctx.war_known & h3_ctx.previous_known_1.fillna(False)&h3_ctx.previous_known_2.fillna(False)
window_count=sum((h3_ctx[f'at_interstate_war_{s}'].fillna(False)|h3_ctx[f'war_previous_{s}'].fillna(False)).astype(int) for s in [1,2])
h3_ctx['war_window_group']=window_count.where(window_known).map({0:'Keines in t-1 oder t',1:'Ein Land in t-1 oder t',2:'Beide in t-1 oder t'}).fillna('Status unbekannt')
all_specs['H3']['df']=h3_ctx

# H8: spätere Wachstumsfenster nur bei vollständig bekannter Kriegsabdeckung.
hh=all_specs['H8']['df']
for lag in range(1,6):
    future=war_country_year[['ccode','year','at_interstate_war','war_known']].copy(); future.year-=lag
    future=future.rename(columns={'at_interstate_war':f'future_war_{lag}','war_known':f'future_known_{lag}'})
    hh=hh.merge(future,on=['ccode','year'],how='left',validate='one_to_one')
fk=hh[[f'future_known_{i}' for i in range(1,6)]].fillna(False).all(axis=1)
fw=hh[[f'future_war_{i}' for i in range(1,6)]].fillna(False).any(axis=1)
hh['future_window_war_group']=np.where(fk,np.where(fw,'Beteiligung im Folgefenster','Keine Beteiligung im Folgefenster'),'Folgefenster unbekannt')
all_specs['H8']['df']=hh

print('Kriegsdatei:',len(raw['wars']),'Beteiligungszeilen;',raw['wars'].WarNum.nunique(),'Kriege.')
print('Bekannter Klassifikationszeitraum:',WAR_CONFIG['war_coverage_start'],'bis',WAR_CONFIG['war_coverage_end'])
print('Überdurchschnittlich häufig: Kriegsjahresanteil >',round(100*frequency_mean_share,3),'Prozent, Länder gleich gewichtet.')
display(frequency_profile.head(12))


### Frei wählbare Stichproben

Beispiele: `waehle_stichprobe('H1', kontinent='Europa', krieg='beteiligt')`
oder `waehle_stichprobe('H3', ausschluss='above_mean_share')`.
Bei einem Kontinent werden für Länderpaare beide Rollen geprüft. Solche
Kontinentteilnahmen überlappen, Kontinentkombinationen sind dagegen exklusiv.
Die Häufigkeitsfilter gelten ausschließlich im gemeinsamen Zeitraum bis 2007.


In [ ]:
def waehle_stichprobe(hypothese, kontinent=None, krieg=None, ausschluss=None, landerliste=None):
    d=all_specs[hypothese]['df'].copy()
    if kontinent is not None:
        if 'ccode' in d: keep=d.continent.eq(kontinent)
        else: keep=d.continent_1.eq(kontinent)|d.continent_2.eq(kontinent)
        d=d.loc[keep].copy()
    if krieg is not None:
        d=d.loc[d.war_known.fillna(False)].copy()
        if krieg=='beteiligt': d=d.loc[d.war_any.eq(1)]
        elif krieg=='nicht_beteiligt': d=d.loc[d.war_any.eq(0)]
        elif krieg=='beide_beteiligt' and 'war_both' in d: d=d.loc[d.war_both.eq(1)]
        else: raise ValueError('krieg: beteiligt, nicht_beteiligt oder beide_beteiligt bei Paaren')
    if ausschluss is not None:
        if ausschluss not in ['above_mean_share','above_p90_share','above_mean_count']:
            raise ValueError('Unbekannte Ausschlussregel')
        d=d.loc[d.war_known.fillna(False)].copy()
        if 'ccode' in d:
            keep=d.frequency_eligible.astype('boolean').fillna(False)&~d[ausschluss].astype('boolean').fillna(True)
        else:
            keep=(d.frequency_eligible_1.astype('boolean').fillna(False)&d.frequency_eligible_2.astype('boolean').fillna(False)&
                ~d[ausschluss+'_1'].astype('boolean').fillna(True)&~d[ausschluss+'_2'].astype('boolean').fillna(True))
        d=d.loc[keep].copy()
    exclude=set(landerliste or [])
    if exclude:
        mask=~d.iso3.isin(exclude) if 'ccode' in d else ~d.iso31.isin(exclude)&~d.iso32.isin(exclude)
        d=d.loc[mask].copy()
    return d


### Statistiken, obere und untere Extremfälle je Gruppe

Für jede Gruppe werden dieselben Roh-, Trim- und Winsor-Korrelationen berechnet.
Extremfälle sind getrennt nach X, Y und vertikalen Regressionsabweichungen
ausgewiesen. Die niedrigsten Nullwerte können echte Beobachtungen sein.
Rangextreme sind nicht automatisch statistische Ausreißer oder Datenfehler.


In [ ]:
group_metrics=[]; extreme_rows=[]; insight_rows=[]; group_registry={}; group_counts={}; cross_records=[]
def context_label(d):
    if 'ccode' in d:
        return d.country_label.astype(str)+' ('+d.iso3.astype(str)+') '+d.year.astype(str)
    return d.country_label_1.astype(str)+' / '+d.country_label_2.astype(str)+' '+d.year.astype(str)

def summarize_group(hyp,kind,label,d,parent_r,overlap=False):
    spec=all_specs[hyp]; x,y=spec['x'],spec['y']
    d=complete(d,[x,y]).reset_index(drop=True)
    gid=f'{hyp}_{len(group_metrics):04d}'
    met={'group_id':gid,'hypothesis':hyp,'grouping':kind,'group':label,
         'overlapping_group':overlap,**corr_metrics(d[[x,y]],x,y)}
    met.update({'x_variable':x,'y_variable':y,'first_year':d.year.min() if len(d) else np.nan,
        'last_year':d.year.max() if len(d) else np.nan,'enough_30_pairs':len(d)>=WAR_CONFIG['group_min_n'],
        'countries':d.ccode.nunique() if 'ccode' in d else pd.unique(d[['ccode1','ccode2']].to_numpy().ravel()).size,
        'entities':d.ccode.nunique() if 'ccode' in d else d.pair_id.nunique(),
        'parent_r':parent_r,'delta_r_parent':met['pearson_r']-parent_r,
        'x_median':d[x].median(),'y_median':d[y].median(),
        'x_min':d[x].min(),'x_max':d[x].max(),'y_min':d[y].min(),'y_max':d[y].max(),
        'y_zero_n':int(d[y].eq(0).sum())})
    if len(d)>=3 and d[x].nunique()>1 and d[y].nunique()>1:
        thresholds={}
        for col in [x,y]:
            lo,hi=d[col].quantile([.01,.99]); q1,q3=d[col].quantile([.25,.75]); med=d[col].median(); mad=(d[col]-med).abs().median()
            thresholds[col]=(lo,hi)
            prefix='x' if col==x else 'y'
            met.update({prefix+'_p01':lo,prefix+'_p99':hi,prefix+'_q25':q1,prefix+'_q75':q3,
                prefix+'_mad':mad,prefix+'_low_ties':int(d[col].eq(d[col].min()).sum()),
                prefix+'_high_ties':int(d[col].eq(d[col].max()).sum())})
        keep=np.ones(len(d),bool); w=d[[x,y]].copy()
        for col in [x,y]:
            lo,hi=thresholds[col]; keep &= d[col].between(lo,hi).to_numpy(); w[col]=w[col].clip(lo,hi)
        trimmed=corr_metrics(d.loc[keep,[x,y]],x,y); winsor=corr_metrics(w,x,y)
        met.update({'trim_n':trimmed['n'],'trim_r':trimmed['pearson_r'],'trim_p_standard':trimmed['p_standard'],
            'trim_removed_pct':100*(1-keep.mean()),'winsor_r':winsor['pearson_r'],
            'trim_sign_flip':np.sign(trimmed['pearson_r'])!=np.sign(met['pearson_r'])})
        # Standardisierung hält die Momentdiagnose auch bei sehr kleinen
        # Anteilsvarianzen numerisch stabil; Pearson r bleibt unverändert.
        scaled=d[list(dict.fromkeys([x,y,'year']+[c for c in ['ccode','ccode1','ccode2'] if c in d]))].copy()
        for col in [x,y]:scaled[col]=(d[col]-d[col].mean())/d[col].std(ddof=0)
        mat=moment_matrix(scaled,x,y); total=mat.sum(axis=0)
        r_del=r_from_moments(total-mat); delta=r_del-met['pearson_r']
        xx=d[x].to_numpy(float); yy=d[y].to_numpy(float)
        slope=np.dot(xx-xx.mean(),yy-yy.mean())/np.dot(xx-xx.mean(),xx-xx.mean())
        residual=yy-(yy.mean()+slope*(xx-xx.mean()))
        d['_residual']=residual; d['_label']=context_label(d)
        d['_delta_r_deleted']=delta
        finite_delta=np.isfinite(delta)
        met['max_row_abs_delta_r']=float(np.max(np.abs(delta[finite_delta]))) if finite_delta.any() else np.nan
        lc=leave_groups(scaled,x,y,'country')
        lc=lc.loc[np.isfinite(lc.delta_r)]
        best=lc.iloc[0] if len(lc) else None
        met.update({'strongest_country':best['country_iso3_default'] if best is not None else None,
                    'country_delta_r':best['delta_r'] if best is not None else np.nan})
        sort_keys=[c for c in ['ccode','ccode1','ccode2','year'] if c in d]
        stable=d.sort_values(sort_keys,kind='stable')
        for dimension,col in [('X',x),('Y',y),('Residuum','_residual')]:
            lower,upper=thresholds.get(col,(np.quantile(residual,.01),np.quantile(residual,.99)))
            q1,q3=np.quantile(d[col],[.25,.75]);iqr=q3-q1
            med=d[col].median();mad=(d[col]-med).abs().median();ties=d[col].value_counts()
            for direction,ascending in [('unten',True),('oben',False)]:
                ix=stable[col].sort_values(ascending=ascending,kind='stable').head(WAR_CONFIG['tail_rows_per_direction']).index
                cases=d.loc[ix]
                for rank,(idx,row) in enumerate(cases.iterrows(),1):
                    val=row[col]
                    target={'group_id':gid,'hypothesis':hyp,'grouping':kind,'group':label,
                        'dimension':dimension,'direction':direction,'rank':rank,'case_label':row['_label'],
                        'year':row.year,'x_value':row[x],'y_value':row[y],'ranked_value':val,
                        'residual_y_units':row['_residual'],'r_without_row':r_del[idx],
                        'delta_r_without_row':row['_delta_r_deleted'],'same_value_n':int(ties.get(val,0)),
                        'percentile_tail_flag':bool(val<lower or val>upper),
                        'iqr_tail_flag':bool(iqr>0 and (val<q1-1.5*iqr or val>q3+1.5*iqr)),
                        'mad_tail_flag':bool(mad>0 and abs(.67448975*(val-med)/mad)>3.5),
                        'group_n':len(d)}
                    for c in ['ccode','iso3','ccode1','ccode2','iso31','iso32','continent','continent_1','continent_2','war_group',
                              'war_names','war_names_1','war_names_2','war_year_share','war_year_share_1','war_year_share_2',
                              'military_growth_pct1','military_growth_pct2','sipri_notes']:
                        if c in d:target[c]=row[c]
                    extreme_rows.append(target)
        high=d.loc[d[y].idxmax()]; low=d.loc[d[y].idxmin()]
        infl=d.iloc[int(np.nanargmax(np.abs(delta)))] if finite_delta.any() else None
        desc=[
            f"Y oben: {high['_label']} mit {high[y]:.5g}; unten: {low['_label']} mit {low[y]:.5g}.",
            f"Nullwerte in Y: {met['y_zero_n']:,}. Bindungen am Minimum: {met['y_low_ties']:,}; am Maximum: {met['y_high_ties']:,}.",
            f"r vollständig {met['pearson_r']:.3f}, getrimmt {met['trim_r']:.3f}, winsorisiert {met['winsor_r']:.3f}; Trimmen entfernt {met['trim_removed_pct']:.1f}%.",
            f"Größter Einfluss einer Zeile: {infl['_label']}, Änderung r beim Weglassen {infl['_delta_r_deleted']:+.4f}." if infl is not None
                else 'Zeileneinfluss nicht bestimmbar: Nach Entfernen bleiben zu wenige Werte oder keine Variation.',
            f"Größter Ländereinfluss: {best['country_iso3_default']}, Änderung r {best['delta_r']:+.4f}; alle Paarrollen entfernt." if best is not None
                else 'Ländereinfluss nicht bestimmbar: Nach Entfernen bleiben zu wenige Werte oder keine Variation.',
        ]
    else:
        desc=['Zu wenige vollständige Beobachtungen oder konstante Variable; keine Korrelations- oder Einflussbewertung.']
        if len(d):
            stable=d.assign(_label=context_label(d)).sort_values(
                [c for c in ['ccode','ccode1','ccode2','year'] if c in d],kind='stable')
            for dimension,col in [('X',x),('Y',y)]:
                lo,hi=d[col].quantile([.01,.99]); q1,q3=d[col].quantile([.25,.75]); iqr=q3-q1
                med=d[col].median(); mad=(d[col]-med).abs().median()
                for direction,ascending in [('unten',True),('oben',False)]:
                    cases=stable.sort_values(col,ascending=ascending,kind='stable').head(WAR_CONFIG['tail_rows_per_direction'])
                    for rank,(_,case) in enumerate(cases.iterrows(),1):
                        value=case[col]
                        extreme_rows.append({'group_id':gid,'hypothesis':hyp,'grouping':kind,'group':label,
                            'dimension':dimension,'direction':direction,'rank':rank,'case_label':case['_label'],
                            'year':case.year,'x_value':case[x],'y_value':case[y],'ranked_value':value,
                            'residual_y_units':np.nan,'r_without_row':np.nan,'delta_r_without_row':np.nan,
                            'same_value_n':int(d[col].eq(value).sum()),'group_n':len(d),
                            'percentile_tail_flag':bool(value<lo or value>hi),
                            'iqr_tail_flag':bool(iqr>0 and (value<q1-1.5*iqr or value>q3+1.5*iqr)),
                            'mad_tail_flag':bool(mad>0 and abs(.67448975*(value-med)/mad)>3.5)})
    if len(d)<WAR_CONFIG['group_min_n']:desc.append('Kleine Gruppe unter 30 Paaren; Kennzahlen nur beschreibend.')
    group_metrics.append(met)
    for i,t in enumerate(desc,1):insight_rows.append({'group_id':gid,'hypothesis':hyp,'grouping':kind,'group':label,'insight_number':i,'insight':t})
    group_registry[gid]={'hypothesis':hyp,'kind':kind,'label':label,'row_index':d.index,
        'metrics':met,'insights':desc,'sample':d[[x,y,'year']].copy()}
    return gid

group_order=defaultdict(list)
for hyp,spec in all_specs.items():
    d,x,y=spec['df'],spec['x'],spec['y']; full=corr_metrics(d,x,y)['pearson_r']
    group_order[hyp].append(summarize_group(hyp,'Gesamt','Gesamt 1950 bis 2014',d,full))
    common=d.loc[d.war_known.fillna(False)]
    cr=corr_metrics(common,x,y)['pearson_r']
    group_order[hyp].append(summarize_group(hyp,'Vergleichszeitraum','Bekannter Kriegsstatus bis 2007',common,cr))
    # Unbekannte Kontinente separat; niemals stillschweigend entfernen.
    for cont in CONTINENTS:
        keep=d.continent.eq(cont) if 'ccode' in d else d.continent_1.eq(cont)|d.continent_2.eq(cont)
        group_order[hyp].append(summarize_group(hyp,'Kontinentteilnahme',cont,d.loc[keep],full,overlap='ccode' not in d))
    if 'ccode' not in d:
        for group,v in d.groupby('continent_pair',dropna=False):
            group_order[hyp].append(summarize_group(hyp,'Kontinentkombination',str(group),v,full))
        for group,v in d.groupby('continent_relation',dropna=False):
            group_order[hyp].append(summarize_group(hyp,'Kontinentrelation',str(group),v,full))
    elif d.continent.eq('Unbekannt').any():
        group_order[hyp].append(summarize_group(hyp,'Kontinentteilnahme','Unbekannt',d.loc[d.continent.eq('Unbekannt')],full))
    for rule,title in [('above_mean_share','Ohne überdurchschnittlichen Kriegsjahresanteil'),
                       ('above_p90_share','Ohne Anteil oberhalb des 90 Perzentils'),
                       ('above_mean_count','Ohne überdurchschnittliche Anzahl Kriege')]:
        sample=waehle_stichprobe(hyp,ausschluss=rule)
        if 'ccode' in common: reference=common.loc[common.frequency_eligible.astype('boolean').fillna(False)]
        else: reference=common.loc[common.frequency_eligible_1.astype('boolean').fillna(False)&common.frequency_eligible_2.astype('boolean').fillna(False)]
        if rule=='above_mean_share':
            group_order[hyp].append(summarize_group(hyp,'Häufigkeitsreferenz',
                f"Bekannter Status und mindestens {WAR_CONFIG['minimum_frequency_years']} beobachtete Jahre",reference,cr))
        reference_r=corr_metrics(reference,x,y)['pearson_r']
        gid=summarize_group(hyp,'Häufigkeitsausschluss',title,sample,reference_r)
        group_registry[gid]['exclusion_rule']=rule; group_order[hyp].append(gid)
    if WAR_CONFIG['manual_exclude_iso3']:
        sample=waehle_stichprobe(hyp,landerliste=WAR_CONFIG['manual_exclude_iso3'])
        group_order[hyp].append(summarize_group(hyp,'Manueller Ausschluss','Ohne konfigurierte Länderliste',sample,full))
    if spec['military']:
        labels=['Keine Beteiligung erfasst','Beteiligung erfasst'] if 'ccode' in d else ['Keines beteiligt','Ein Land beteiligt','Beide beteiligt']
        for group in labels+['Status unbekannt']:
            v=d.loc[d.war_group.eq(group)]
            group_order[hyp].append(summarize_group(hyp,'Kriegsstatus',group,v,cr if group!='Status unbekannt' else full))
        if WAR_CONFIG['include_crossed_war_continent']:
            for cont in CONTINENTS:
                keep=common.continent.eq(cont) if 'ccode' in common else common.continent_1.eq(cont)|common.continent_2.eq(cont)
                c=common.loc[keep]; parent=corr_metrics(c,x,y)['pearson_r']
                for group in labels:
                    gid=summarize_group(hyp,'Kontinent und Krieg',cont+' / '+group,c.loc[c.war_group.eq(group)],parent,'ccode' not in d)
                    group_order[hyp].append(gid)
                    cross_records.append({'hypothesis':hyp,'continent':cont,'war_status':group,'group_id':gid})
    if hyp=='H3':
        for group,v in d.groupby('war_window_group'):
            group_order[hyp].append(summarize_group(hyp,'Krieg im Wachstumsintervall',group,v,cr))
    if hyp=='H8':
        for group,v in d.groupby('future_window_war_group'):
            group_order[hyp].append(summarize_group(hyp,'Krieg im Folgefenster',group,v,full))
    print(hyp,':',len(group_order[hyp]),'Gruppen ausgewertet.')

group_results=pd.DataFrame(group_metrics)
group_extremes=pd.DataFrame(extreme_rows)
group_insights=pd.DataFrame(insight_rows)


### Modelle mit Kriegsstatus und Kontinenten

Die Modelle bleiben explorativ. Länder- und Jahreseffekte berücksichtigen
zeitkonstante Länderunterschiede und gemeinsame Jahresunterschiede. Interaktionen
prüfen, ob sich die Steigung mit erfasster Kriegsbeteiligung unterscheidet.
Die Querschnitts-/Gruppenkorrelationen und die Modellkoeffizienten beantworten
unterschiedliche Fragen. Paar-/Jahrescluster erfassen nicht alle Abhängigkeiten
verschiedener Paare mit demselben Land. Kleine oder kollineare Modelle werden
mit Begründung übersprungen, nicht als Nullergebnis ausgegeben.


In [ ]:
war_model_rows=[]; war_model_details=[]; war_model_status=[]
def model_sample(hyp,d):
    spec=all_specs[hyp]; x,y=spec['x'],spec['y']; out=d.copy()
    if hyp=='H1':out['x_model']=np.log(out[x].where(out[x].gt(0))); unit='log real GDP per worker'
    else:out['x_model']=out[x]; unit=x
    out['y_model']=out[y]
    return out,unit

def run_context_model(hyp,label,d,include_war=False,clip=False):
    spec=all_specs[hyp]; x,y=spec['x'],spec['y']
    d=d.copy()
    if clip and len(d):
        for col in [x,y]: d[col]=d[col].clip(*d[col].quantile([.01,.99]))
    d,unit=model_sample(hyp,d)
    xx=['x_model']
    if include_war:
        d=d.loc[d.war_known.fillna(False)].copy()
        d['x_centered']=d.x_model-d.x_model.mean()
        d['war_any']=pd.to_numeric(d.war_any,errors='coerce').astype(float)
        d['war_x']=d.war_any*d.x_centered
        xx+=['war_any','war_x']
        if 'ccode' not in d:
            d['war_both']=pd.to_numeric(d.war_both,errors='coerce').astype(float)
            d['both_x']=d.war_both*d.x_centered
            xx+=['war_both','both_x']
    if hyp=='H2':
        d['log_population']=np.log(d.tpop.where(d.tpop.gt(0)))
        d['log_productivity']=np.log(d.productivity.where(d.productivity.gt(0)))
        xx+=['log_population','log_productivity']
    entity='ccode' if 'ccode' in d else 'pair_id'
    name=hyp+' '+label+(' winsor' if clip else '')
    complete_n=len(complete(d,['y_model']+xx))
    try:
        if complete_n<max(100,len(xx)*20):raise ValueError('Too few complete observations for this model')
        rows,details=fit_absorbed(name,d,'y_model',xx,entity,True,
            'Exploratory; no causal identification; shared-country dyadic dependence may remain' if entity=='pair_id'
            else 'Exploratory country/year clustered inference; no causal identification')
        for row in rows:row.update(hypothesis=hyp,group=label,x_model_definition=unit,war_interaction=include_war,winsorised=clip)
        details.update(hypothesis=hyp,group=label,x_model_definition=unit,war_interaction=include_war,winsorised=clip)
        war_model_rows.extend(rows); war_model_details.append(details)
        status='computed'; reason=''
    except (ValueError,np.linalg.LinAlgError,RuntimeError) as exc:
        status='skipped'; reason=str(exc)
    war_model_status.append({'hypothesis':hyp,'model':name,'n_input':len(d),'n_complete_candidates':complete_n,'status':status,'reason':reason})

for hyp,spec in all_specs.items():
    if not spec['military']:continue
    d=spec['df']; known=d.loc[d.war_known.fillna(False)].copy()
    run_context_model(hyp,'Gleicher Kriegszeitraum ohne Kriegsregressoren',known)
    run_context_model(hyp,'Kriegsstatus mit Interaktion',known,True)
    run_context_model(hyp,'Kriegsstatus mit Interaktion',known,True,True)
    run_context_model(hyp,'Ohne überdurchschnittlichen Kriegsjahresanteil',waehle_stichprobe(hyp,ausschluss='above_mean_share'))
    if 'ccode' in known: reference=known.loc[known.frequency_eligible.astype('boolean').fillna(False)]
    else: reference=known.loc[known.frequency_eligible_1.astype('boolean').fillna(False)&known.frequency_eligible_2.astype('boolean').fillna(False)]
    run_context_model(hyp,'Häufigkeitsreferenz vor Länderausschluss',reference)
    for cont in CONTINENTS:
        s=waehle_stichprobe(hyp,kontinent=cont)
        run_context_model(hyp,'Kontinent '+cont,s)
        if WAR_CONFIG['include_crossed_war_continent']:
            run_context_model(hyp,'Kontinent '+cont+' mit Kriegsstatus',s.loc[s.war_known.fillna(False)],True)
context_models=pd.DataFrame(war_model_rows)
context_model_details=pd.DataFrame(war_model_details)
context_model_status=pd.DataFrame(war_model_status)
print('Zusatzmodelle:',context_model_status.status.value_counts().to_dict())


### Grafiken je Hypothese und Gruppierung

Ein Diagnosebild zeigt die Verteilung und den Einfluss oberer/unterer Fälle.
Bei Kontinentkombinationen und gekreuzten Gruppen werden Vergleichsgrafiken
verwendet; die vollständigen Kennzahlen und individuellen Fälle stehen in den
zugehörigen Tabellen und direkt unter der Vergleichsgrafik.


In [ ]:
HYP_TITLES={
 'H1':'Produktivität und Militärpersonal','H2':'Importkonzentration und Militärpersonal',
 'H3':'Handelsverflechtung und Militärausgabenwachstum','H4':'Wohlstandsunterschied und Handel',
 'H8':'Exportdistanz und späteres Wachstum','E1':'Bevölkerung und Importkonzentration',
 'E2':'Urbanisierung und Militärpersonal','E3':'Wirtschaftsgröße und absolute Asymmetrie',
 'E4':'Wirtschaftsgröße und relative Asymmetrie','E5':'Entfernung und Handel'}
LABELS.update({'tpop':'Bevölkerung (Tausend)','urban_share':'Städtischer Bevölkerungsanteil (0-1)',
    'asymmetry_relative':'Relative Asymmetrie (0-1)','log_distance':'Logarithmierte Entfernung (km)'})
context_charts=[]
def context_figure(hyp,section,title,fig,gids,notes=None):
    buf=BytesIO(); fig.savefig(buf,format='png',dpi=WAR_CONFIG['chart_dpi'],bbox_inches='tight',facecolor='white')
    context_charts.append({'hypothesis':hyp,'section':section,'title':title,'png':buf.getvalue(),
                           'group_ids':gids,'notes':notes or []})
    plt.close(fig)

def individual_group_chart(gid):
    rec=group_registry[gid]; hyp=rec['hypothesis']; spec=all_specs[hyp]; x,y=spec['x'],spec['y']; m=rec['metrics']
    d=rec['sample']; title=rec['label']
    fig,axs=plt.subplots(1,2,figsize=(12.6,4.5),layout='constrained')
    density(axs[0],d,x,y,'Verteilung der vollständigen Wertepaare')
    t=group_extremes.loc[group_extremes.group_id.eq(gid)&group_extremes.dimension.eq('Residuum')]
    if len(t):
        t=t.sort_values(['direction','rank']).drop_duplicates(['case_label','ranked_value'])
        lab=[textwrap.shorten(str(s),width=40,placeholder='...') for s in t.case_label]
        pos=np.arange(len(t)); colors=np.where(t.ranked_value.ge(0),'#ae563e','#365e83')
        axs[1].barh(pos,t.ranked_value,color=colors); axs[1].set_yticks(pos,lab,fontsize=8)
        axs[1].axvline(0,color='grey',lw=.7)
        axs[1].set(xlabel='Abweichung von einfacher Regressionsgerade (Y-Einheit)',title='Individuelle Abweichungen nach oben und unten')
    else:axs[1].text(.5,.5,'Keine Einflussdiagnose verfügbar',ha='center',transform=axs[1].transAxes)
    fig.suptitle(hyp+' '+title,fontsize=12)
    context_figure(hyp,rec['kind'],title,fig,[gid])

for hyp in all_specs:
    entries=[group_registry[g] for g in group_order[hyp]]
    # Alle marginalen Kontinente, Kriegsklassen und Ausschlussvarianten erhalten
    # eigene Abbildungen. Sehr kleine Gruppen bleiben als solche gekennzeichnet.
    detailed={'Gesamt','Vergleichszeitraum','Kontinentteilnahme','Kriegsstatus','Häufigkeitsreferenz','Häufigkeitsausschluss',
              'Manueller Ausschluss','Krieg im Wachstumsintervall','Krieg im Folgefenster'}
    for gid in group_order[hyp]:
        rec=group_registry[gid]
        if rec['kind'] in detailed:individual_group_chart(gid)
    if 'ccode' not in all_specs[hyp]['df']:
        ids=[g for g in group_order[hyp] if group_registry[g]['kind']=='Kontinentkombination']
        a=group_results.loc[group_results.group_id.isin(ids)]
        mat=np.full((6,6),np.nan); counts=np.zeros((6,6),int)
        for row in a.itertuples():
            parts=row.group.split(' / ')
            if len(parts)==2 and all(p in CONTINENTS for p in parts):
                i,j=sorted([CONTINENTS.index(p) for p in parts]); mat[i,j]=row.pearson_r; counts[i,j]=row.n
        fig,ax=plt.subplots(figsize=(9.5,6),layout='constrained')
        im=ax.imshow(np.ma.masked_invalid(mat),vmin=-1,vmax=1,cmap='RdBu_r')
        for i in range(6):
            for j in range(i,6):
                ax.text(j,i,f'r={mat[i,j]:.3f}\nn={counts[i,j]:,}' if np.isfinite(mat[i,j]) else f'n={counts[i,j]:,}',
                    ha='center',va='center',fontsize=8,color='white' if np.isfinite(mat[i,j]) and abs(mat[i,j])>.6 else 'black')
        ax.set_xticks(range(6),CONTINENTS,rotation=25,ha='right'); ax.set_yticks(range(6),CONTINENTS)
        ax.set_title(hyp+' Exklusive Kontinentkombinationen'); fig.colorbar(im,ax=ax,label='Pearson r',shrink=.75)
        context_figure(hyp,'Kontinentkombination','Alle Kontinentkombinationen',fig,ids,
            ['Jedes Länderpaar steht in genau einer Kombination. Diagonale: gleicher Kontinent.'])
    cross=[c for c in cross_records if c['hypothesis']==hyp]
    if cross:
        fig,axs=plt.subplots(2,3,figsize=(13,7.5),sharey=True,layout='constrained')
        rvals=group_results.set_index('group_id').loc[[c['group_id'] for c in cross],'pearson_r'].dropna()
        lower=min(0,rvals.min()) if len(rvals) else 0
        upper=max(0,rvals.max()) if len(rvals) else 0
        pad=max(.025,.12*(upper-lower))
        for ax,cont in zip(axs.ravel(),CONTINENTS):
            ids=[c['group_id'] for c in cross if c['continent']==cont]
            v=group_results.set_index('group_id').loc[ids]
            pos=np.arange(len(v))
            bars=ax.bar(pos,v.pearson_r,color='#365e83')
            labels=[textwrap.fill(s.split(' / ',1)[1],15)+'\nn='+format(int(n),',') for s,n in zip(v.group,v.n)]
            ax.set_xticks(pos,labels,fontsize=8);ax.set_ylim(lower-pad,upper+pad)
            ax.axhline(0,color='grey',lw=.7); ax.set(title=cont,ylabel='Pearson r')
            for bar,n in zip(bars,v.n):
                if n<WAR_CONFIG['group_min_n']:bar.set_hatch('//');bar.set_alpha(.5)
        context_figure(hyp,'Kontinent und Krieg','Kriegsstatus innerhalb der Kontinente',fig,
            [c['group_id'] for c in cross],['Rohkorrelationen bei gleicher Y-Skala. Schraffiert: kleine Gruppe unter 30; fehlender Balken: keine berechenbare Korrelation. Gruppen können sich in Ländern und Jahren unterscheiden.'])
    if all_specs[hyp]['military'] and len(context_models):
        v=context_models.loc[context_models.hypothesis.eq(hyp)&
            context_models.group.eq('Kriegsstatus mit Interaktion')&
            context_models.term.isin(['x_model','war_x','both_x'])].copy()
        terms=[t for t in ['x_model','war_x','both_x'] if t in set(v.term)]
        if terms:
            fig,axs=plt.subplots(1,len(terms),figsize=(4.4*len(terms),3.4),squeeze=False,layout='constrained')
            titles={'x_model':'Steigung ohne Beteiligung','war_x':'Änderung bei mind. einem beteiligten Land',
                    'both_x':'Zusätzliche Änderung bei zwei beteiligten Ländern'}
            for ax,term in zip(axs.ravel(),terms):
                part=v.loc[v.term.eq(term)].sort_values('winsorised')
                yy=np.arange(len(part))
                ax.scatter(part.coefficient,yy,color='#365e83')
                for i,(_,r) in enumerate(part.iterrows()):
                    if np.isfinite(r.ci_low) and np.isfinite(r.ci_high):
                        ax.plot([r.ci_low,r.ci_high],[i,i],color='#365e83')
                ax.set_yticks(yy,['Winsor 1/99%' if w else 'Vollständige Werte' for w in part.winsorised],fontsize=8)
                ax.set_ylim(-.75,len(part)-.25)
                ax.axvline(0,color='grey',lw=.7);ax.set(title=textwrap.fill(titles[term],32),xlabel='Koeffizient und exploratives 95%-Intervall')
            context_figure(hyp,'Kriegsmodell mit Länder-/Paar- und Jahreseffekten','Kriegsinteraktionen: vollständige Werte und Winsor-Sensitivität',fig,[],
                ['war_x ist eine Steigungsänderung; bei Paaren kommt both_x zusätzlich hinzu. Keine Kausalaussage. Beide beteiligt bedeutet nicht gegeneinander.',
                 'H1: X logarithmiert. H2: zusätzlich Bevölkerung und Produktivität kontrolliert. Modelle mit unbestimmtem Standardfehler zeigen nur den Punktschätzer.'])
            context_charts[-1]['model_frame']=v[['term','winsorised','coefficient','ci_low','ci_high','p_cluster_exploratory','n']]
    print(hyp,': Grafiken fertig.')

# Bisherige Abbildungen im selben Dokument je Hypothese erhalten.
for old in chart_items:
    sheet=old['sheet']
    if sheet.startswith('Charts_H'):hyp=sheet.replace('Charts_','')
    elif sheet=='Charts_differences':hyp='H1'
    elif sheet=='Charts_influence':hyp='H3'
    else:hyp='H4'
    root_gid=group_order[hyp][0]
    context_charts.append({'hypothesis':hyp,'section':'Bisherige vertiefende Analysen','title':old['title'],
        'png':old['png'],'group_ids':[root_gid],
        'notes':['Weitere Zahlen der Teilabbildungen stehen in den Alt Tabellen, insbesondere Regression und Trade_groups.']})


### Ein strukturierter Bericht mit Zeitstempel

Die Berichtsblätter sind nach Hypothese geordnet. Gruppentitel führen zur Grafik,
darunter folgen n, r, rho, p sowie knappe Fallhinweise. Für jede Hypothese gibt es
zusätzlich filterbare Gruppen-, Extremfall- und Insight-Tabellen. Quellen und
bisherige Resultate werden unverändert als Alt Tabellen mitgeführt.


In [ ]:
from openpyxl.worksheet.pagebreak import Break

def typed_excel(value):
    if value is None or value is pd.NA:return None
    if isinstance(value,np.generic):value=value.item()
    if isinstance(value,float) and not np.isfinite(value):return None
    if isinstance(value,(list,dict,tuple)):return str(value)
    if pd.isna(value):return None
    if isinstance(value,str) and value.startswith(('=','+','-','@')):return "'"+value
    return value

def write_frame(wb,name,df):
    if len(df)>1_048_575:raise ValueError(name+': Excel-Zeilenlimit überschritten')
    ws=wb.create_sheet(name[:31]); ws.sheet_view.showGridLines=False
    ws.append(list(df.columns))
    for row in df.itertuples(index=False,name=None):ws.append([typed_excel(v) for v in row])
    ws.freeze_panes='B2'; ws.auto_filter.ref=ws.dimensions
    for j,col in enumerate(df.columns,1):
        c=ws.cell(1,j); c.font=Font(name='Arial',size=10,bold=True,color='FFFFFF'); c.fill=PatternFill('solid',fgColor='243952')
        c.alignment=Alignment(wrap_text=True,horizontal='center',vertical='center')
        vals=df[col].dropna().astype(str).head(100)
        width=min(55,max(13,len(str(col))+2,max([0]+[len(v)+2 for v in vals])))
        ws.column_dimensions[get_column_letter(j)].width=width
        for row in ws.iter_rows(min_row=2,min_col=j,max_col=j):
            cell=row[0]; cell.font=Font(name='Arial',size=10,color='182638')
            if isinstance(cell.value,float):cell.number_format='0.000E+00' if col.startswith('p_') or col.endswith('_p_standard') else '0.0000'
            if isinstance(cell.value,str) and len(cell.value)>width-2:
                cell.alignment=Alignment(wrap_text=True,vertical='top')
                ws.row_dimensions[cell.row].height=min(409,max(ws.row_dimensions[cell.row].height or 18,14*math.ceil(len(cell.value)/(max(width-2,1)*.85))+4))
    ws.row_dimensions[1].height=42
    return ws

workbook=openpyxl.Workbook(); workbook.remove(workbook.active)
start_sheet=workbook.create_sheet('Start'); start_sheet.sheet_view.showGridLines=False
start_sheet.column_dimensions['A'].width=3; start_sheet.column_dimensions['B'].width=50; start_sheet.column_dimensions['C'].width=100
start_sheet['B2']='Kriegsstatus und Kontinente'; start_sheet['B2'].font=Font(name='Arial',size=16,bold=True)
intro=[
 ('Zeitraum','Hauptanalysen 1950-2014; Kriegsstatus und Häufigkeitsfilter nur bei bekannter COW-Abdeckung bis 2007.'),
 ('Kriegsbegriff','Erfasste Beteiligung an zwischenstaatlichem Krieg. Bürgerkriege fehlen. Keine erfasste Beteiligung bedeutet nicht allgemein Frieden.'),
 ('Häufigkeitsgrenze',f'Kriegsjahresanteil > {frequency_mean_share:.6f}, Referenzländer gleich gewichtet, mindestens {WAR_CONFIG["minimum_frequency_years"]} beobachtete Jahre.'),
 ('Einzelne Datei','Alle alten und neuen Grafiken sowie Kennzahlen sind in dieser Datei. Pro Hypothese zuerst das Blatt Bericht öffnen.'),
 ('Extremfälle','Obere/untere Werte von X, Y und Regressionsabweichungen; Rangextreme sind nicht automatisch Datenfehler.'),
 ('Abhängigkeit','n zählt Zeilen. Standard-p-Werte nehmen unabhängige Beobachtungen an; hier ist das nicht erfüllt. Gruppenvergleiche und Zusatzmodelle sind explorativ.'),
 ('Kontinentteilnahme','Bei Länderpaaren kann eine Beziehung in zwei Kontinentteilnahmen vorkommen. Kontinentkombinationen sind exklusiv.'),
]
for i,(label,text) in enumerate(intro,4):
    start_sheet.cell(i,2,label).font=Font(name='Arial',size=10,bold=True)
    start_sheet.cell(i,3,text).alignment=Alignment(wrap_text=True,vertical='top')
    start_sheet.row_dimensions[i].height=34
for i,hyp in enumerate(all_specs,13):
    c=start_sheet.cell(i,2,hyp+' '+HYP_TITLES[hyp]); c.hyperlink=f"#'{hyp}_Bericht'!B2"; c.font=Font(name='Arial',size=11,color='225980',underline='single')
    c.alignment=Alignment(wrap_text=True,vertical='center'); start_sheet.row_dimensions[i].height=32
    start_sheet.cell(i,3,'Grafiken und kurze Zahlen, anschließend Gruppen, Extremfälle und Insights als filterbare Tabellen.')

metric_header=['Gruppe','n','Länder','r','rho','p Standard','r getrimmt','r winsor']
report_index=[]
def style_metric_row(ws,row,values,header=False):
    for j,val in enumerate(values,2):
        c=ws.cell(row,j,typed_excel(val)); c.font=Font(name='Arial',size=9,bold=header,color='FFFFFF' if header else '17283A')
        if header:c.fill=PatternFill('solid',fgColor='243952')
        c.alignment=Alignment(wrap_text=True,vertical='center')
        if not header and isinstance(val,float):c.number_format='0.000E+00' if j==7 else '0.0000'
    ws.row_dimensions[row].height=max(30,14*math.ceil(len(str(values[0]))/35))

for hyp in all_specs:
    ws=workbook.create_sheet(hyp+'_Bericht'); ws.sheet_view.showGridLines=False
    ws.sheet_properties.tabColor='243952'
    ws.column_dimensions['A'].width=3; ws.column_dimensions['B'].width=38
    for col in range(3,13):ws.column_dimensions[get_column_letter(col)].width=13
    ws['B2']=hyp+' '+HYP_TITLES[hyp]; ws['B2'].font=Font(name='Arial',size=15,bold=True)
    ws['B3']='Grafik, Kennzahlen und individuelle Hinweise je Gruppe. Standard-p-Werte nur eingeschränkt interpretieren.'
    ws.merge_cells('B3:L3'); ws['B3'].alignment=Alignment(wrap_text=True); ws.row_dimensions[3].height=30
    ws['B4']='Zur Übersicht'; ws['B4'].hyperlink="#'Start'!B2"
    row=7; section_seen=None
    for item in [c for c in context_charts if c['hypothesis']==hyp]:
        if item['section']!=section_seen:
            section_seen=item['section']; ws.cell(row,2,section_seen).font=Font(name='Arial',size=13,bold=True); row+=2
        header_row=row
        ws.merge_cells(start_row=row,start_column=2,end_row=row,end_column=12)
        ws.cell(row,2,item['title']).font=Font(name='Arial',size=11,bold=True)
        ws.cell(row,2).alignment=Alignment(wrap_text=True); ws.row_dimensions[row].height=30; row+=2
        img=ExcelImage(BytesIO(item['png']))
        scale=1050/img.width; img.width*=scale; img.height*=scale
        ws.add_image(img,f'B{row}')
        image_rows=math.ceil(img.height/20)+1
        for rr in range(row,row+image_rows):ws.row_dimensions[rr].height=15
        row+=image_rows+1
        caption_header=['Term','Winsor','Koeff.','95% unten','95% oben','p Cluster','n'] if 'model_frame' in item else metric_header
        style_metric_row(ws,row,caption_header,True); row+=1
        for gid in item['group_ids']:
            m=group_registry[gid]['metrics']
            style_metric_row(ws,row,[m['group'],m['n'],m['countries'],m['pearson_r'],m['spearman_rho'],m['p_standard'],m.get('trim_r'),m.get('winsor_r')]);row+=1
            report_index.append({'hypothesis':hyp,'section':item['section'],'chart_title':item['title'],
                'group_id':gid,'sheet':ws.title,'chart_heading_row':header_row,'metrics_row':row-1})
        if 'model_frame' in item:
            for vals in item['model_frame'].itertuples(index=False,name=None):
                style_metric_row(ws,row,vals);row+=1
            report_index.append({'hypothesis':hyp,'section':item['section'],'chart_title':item['title'],
                'group_id':'','sheet':ws.title,'chart_heading_row':header_row,
                'metrics_row':row-len(item['model_frame'])})
        notes=list(item['notes'])
        if len(item['group_ids'])==1:notes+=group_registry[item['group_ids'][0]]['insights']
        elif item['group_ids']:notes+=['Individuelle obere/untere Fälle jeder enthaltenen Gruppe stehen im Blatt '+hyp+'_Extreme; group_id verbindet Grafik und Tabellen.']
        for text in notes:
            ws.merge_cells(start_row=row,start_column=2,end_row=row,end_column=12)
            c=ws.cell(row,2,text); c.font=Font(name='Arial',size=9); c.alignment=Alignment(wrap_text=True,vertical='top')
            ws.row_dimensions[row].height=max(18,14*math.ceil(len(text)/140)); row+=1
        row+=3
    ws.freeze_panes='C5'; ws.sheet_properties.pageSetUpPr.fitToPage=True
    ws.page_setup.orientation='landscape';ws.page_setup.paperSize=ws.PAPERSIZE_A3
    ws.page_setup.fitToWidth=1;ws.page_setup.fitToHeight=0
    ws.print_options.horizontalCentered=True
    write_frame(workbook,hyp+'_Gruppen',group_results.loc[group_results.hypothesis.eq(hyp)])
    write_frame(workbook,hyp+'_Extreme',group_extremes.loc[group_extremes.hypothesis.eq(hyp)])
    write_frame(workbook,hyp+'_Insights',group_insights.loc[group_insights.hypothesis.eq(hyp)])

write_frame(workbook,'Grafikindex',pd.DataFrame(report_index))
write_frame(workbook,'Kriegsprofile',frequency_profile)
write_frame(workbook,'Kriegsjahre',war_country_year.loc[war_country_year.year.between(START_YEAR-1,END_YEAR)])
write_frame(workbook,'Kriegsbeteiligung_Jahr',war_participation_years.loc[war_participation_years.year.between(START_YEAR-1,END_YEAR)])
write_frame(workbook,'Kriegsepisoden',war_episodes)
write_frame(workbook,'Datumsprobleme',war_date_issues)
write_frame(workbook,'Kontinentzuordnung',continent_lookup)
write_frame(workbook,'Zusatzmodelle',context_models)
write_frame(workbook,'Modelldetails',context_model_details)
write_frame(workbook,'Modellstatus',context_model_status)

context_methods=[
 ('Kriegsabdeckung','COW Inter-State War v4.0 covers 1816-2007. The last positive episode year is not the end of source coverage. Years 2008+ are unknown, never peace.'),
 ('Definition','Any participation in a recorded interstate war during the calendar year. It may be fought abroad. Civil, extra-state and non-state wars are not covered.'),
 ('Episoden','Both separate participation episodes are expanded independently to years; gaps are not bridged. Multiple wars in one year count as one war-year but several distinct wars.'),
 ('Datumscodes','-8: no second episode; -7: ongoing to end of source coverage; unknown year bounds produce unknown possible country-years, not zero.'),
 ('Länderpaare','No/one/both countries participating in ANY recorded interstate war. Both participating does not imply enemies, allies, same war, or direct fighting against each other.'),
 ('Kriegsfreie Gruppe','No recorded interstate war participation within source coverage; not general peace. Angola civil-war uncertainty is not removed by this filter.'),
 ('Historische Identität','War flags join on COW country code and exact year. Frequency profiles use eligible ISO identities, preserving predecessor/successor exclusions from preparation.'),
 ('Vergleichszeitraum','Current-war comparisons use only known status to 2007. Frequency-exclusion variants additionally require at least 10 observed years for every country, and compare with a separate same-period frequency-eligible reference.'),
 ('Häufigkeitsanteil','Unique years with participation / eligible observed member-years, 1950-2007. Reference mean gives each country equal weight and includes zero-war countries with at least 10 observed years.'),
 ('Häufigkeitsfilter','Primary: above mean war-year share. Additional: strictly above 90th percentile share; above mean distinct-war count. Short histories have unknown eligibility and are excluded from filtered variants.'),
 ('Rückblick','Frequency profiles use the full observation period retrospectively; these are descriptive selection comparisons, not forecasts or causal effects.'),
 ('H3 Intervall','Additional status for t-1 or t because the spending growth rate uses both years. Current-year classification is retained separately.'),
 ('H8 Fenster','Additional future-window status requires all t+1 to t+5 years known; otherwise unknown. Overlapping growth windows still create dependence.'),
 ('Kontinente','CEPII static continent assignment; America split into North America including Central America/Caribbean and explicit South America ISO list; Pacific renamed Oceania. Antarctica has no eligible sovereign-country observations.'),
 ('Transkontinental','Each country gets one CEPII-based continent. Different classifications are a possible sensitivity, not silently duplicated assignments.'),
 ('Paarkontinente','Unordered continent pairs partition known data. Participation in a continent includes either role and overlaps for cross-continent pairs. Do not sum these rows as independent samples.'),
 ('Extremfälle','Top/bottom three ranks for X, Y and simple-OLS residuals per group. Ties are explicitly counted; returned rows are a deterministic selection. Extreme rank is not proof of error.'),
 ('Ausreißersensitivität','Within each group: 1st/99th percentile trim and winsor, with dropped counts; 1.5 IQR and MAD z>3.5 flags on selected cases. True zeros retained.'),
 ('Einfluss','Exact Pearson r after deleting a row; country deletion removes every row involving that country. These are diagnostics, not permission to delete observations.'),
 ('Kleine Gruppen','Correlations require at least three pairs and variable variation. Groups below 30 are marked; no automatic acceptance/rejection. The course 5000-pair rule applies to main hypotheses, not each additional split.'),
 ('p Werte','Conventional two-sided Pearson p-values ignore panel/dyad dependence. Many post-hoc subgroup comparisons are exploratory. Zero p-values can be numerical underflow.'),
 ('Modelle','Absorbed entity/year effects; interactions with any/both participation. H1 uses log productivity. H2 additionally controls log population/productivity. Cluster covariance: entity+year minus intersection.'),
 ('Modellgrenzen','No causal interpretation. Pair/year clustering misses some dependence between pairs sharing a country. Skipped models and reasons are retained. Raw slope differences may reflect composition.'),
 ('Bericht','One timestamped static XLSX. Re-run notebook to refresh. Charts are stored within hypothesis report sheets with numeric tables directly below. Full case and insight tables link via group_id.'),
]
write_frame(workbook,'Methodik_Erweiterung',pd.DataFrame(context_methods,columns=['topic','definition']))
parameters=pd.DataFrame([{'parameter':k,'value':str(v)} for k,v in WAR_CONFIG.items()]+[
 {'parameter':'frequency_mean_share','value':str(frequency_mean_share)},
 {'parameter':'frequency_q90_share','value':str(frequency_q90_share)},
 {'parameter':'frequency_mean_distinct_wars','value':str(frequency_mean_count)},
 {'parameter':'war_extension_code_sha256','value':WAR_CODE_SHA256},
 {'parameter':'old_extension_code_sha256','value':ANALYSIS_CODE_SHA256},
 {'parameter':'source_wars_url','value':'https://correlatesofwar.org/data-sets/COW-war/'},
 {'parameter':'war_codebook_url','value':'https://correlatesofwar.org/wp-content/uploads/Inter-StateWars_Codebook.pdf'},
])
write_frame(workbook,'Parameter_Erweiterung',parameters)
for name,df in report_tables.items():write_frame(workbook,('Alt_'+name)[:31],df)

output_dir=PROJECT_DIR/'Ergebnisse';output_dir.mkdir(exist_ok=True)
stamp=datetime.now(ZoneInfo('Europe/Berlin')).strftime('%Y-%m-%d_%H-%M-%S_%f_%z')
context_report_path=output_dir/f'Analysebericht_Krieg_Kontinente_{stamp}.xlsx'
with context_report_path.open('xb') as stream:workbook.save(stream)
print('FERTIG:',context_report_path.resolve())
print('Einzelne Datei:',len(context_charts),'Grafiken;',len(group_results),'Gruppen;',len(group_extremes),'Extremfallzeilen.')
